In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp "/content/drive/MyDrive/MIMIC IV.zip" /content/mimic_iv.zip
!unzip -q /content/mimic_iv.zip -d /content/mimic
!rm /content/mimic_iv.zip   # frees ~10 GB of disk after extraction

In [ ]:
!find /content/mimic -maxdepth 4 | head -50

In [ ]:
!apt-get -qq install unrar
!cp "/content/drive/MyDrive/eicu-collaborative-research-database-2.0.rar" /content/eicu.rar
!unrar x -idq /content/eicu.rar /content/eicu/
!rm /content/eicu.rar
!df -h /content

In [ ]:
!find /content/eicu -type f | sort | head -80
!find /content/eicu -type f -name "*.gz" -exec ls -lh {} \; | awk '{print $5, $9}'

In [ ]:
!find /content/mimic/mimiciv -name "*.csv.gz" | sort

In [ ]:
import os
EICU = '/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0'
expected = [l.split()[-1] for l in open(f'{EICU}/SHA256SUMS.txt')]
missing = [f for f in expected if not os.path.exists(f'{EICU}/{f}')]
print(len(expected), 'expected;', len(missing), 'missing')
print(missing)

In [ ]:
!ls -lh "/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0.zip"
!unzip -l "/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0.zip" | grep -i -E "patient|vital|treatment"

In [ ]:
Z = "/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0.zip"
!ls -lh "{Z}"
!unzip -tq "{Z}" | tail -3
!unzip -l "{Z}" | grep -i -E "patient|vitalPeriodic"

In [ ]:
Z = "/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0.zip"
!unzip -n -q "{Z}" "*/patient.csv.gz" "*/vitalPeriodic.csv.gz" "*/vitalAperiodic.csv.gz" -d /content/eicu_extra
!find /content/eicu_extra -name "*.gz" -exec ls -lh {} \;

In [ ]:
import os, glob, shutil, subprocess

SRC   = '/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0'
EXTRA = '/content/eicu_extra/eicu-collaborative-research-database-2.0'

r = subprocess.run(
    f'cd "{EXTRA}" && grep -E " (patient|vitalPeriodic|vitalAperiodic)\\.csv\\.gz" "{SRC}/SHA256SUMS.txt" | sha256sum -c',
    shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr)

if r.stdout.count(': OK') == 3:
    for f in glob.glob(f'{EXTRA}/*.csv.gz'):
        shutil.move(f, f'{SRC}/{os.path.basename(f)}')
    zip_path = os.path.join(os.path.dirname(SRC), 'eicu-collaborative-research-database-2.0.zip')
    if os.path.exists(zip_path):
        os.remove(zip_path)
    print('Done:', len(glob.glob(f'{SRC}/*.csv.gz')), 'eICU tables ready')
else:
    print('Checksum problem, nothing was moved')

In [ ]:
!mv "/content/mimic/MIMIC .vi3" /content/mimic/mimiciv
!find /content/mimic/mimiciv -name "*.csv.gz" | sed 's#.*/##' | sort

In [ ]:
d = '/content/mimic/mimiciv/Hosp'
for t in ['prescriptions', 'microbiologyevents']:
    !mkdir -p "{d}/{t}.csv"
    !cp "/content/drive/MyDrive/{t}.csv.gz" "{d}/{t}.csv/{t}.csv.gz"
!ls -lh {d}/prescriptions.csv {d}/microbiologyevents.csv

In [ ]:
d = '/content/mimic/mimiciv/Hosp'
for t in ['prescriptions', 'microbiologyevents']:
    !mkdir -p "{d}/{t}.csv"
    !cp "/content/drive/MyDrive/{t}.csv.gz" "{d}/{t}.csv/{t}.csv.gz"
!ls -lh {d}/prescriptions.csv {d}/microbiologyevents.csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
checks = {
 "MIMIC folder (unzipped)":      "/content/mimic/mimiciv",
 "eICU folder (unzipped)":       "/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0",
 "prescriptions in Drive":       "/content/drive/MyDrive/prescriptions.csv.gz",
 "microbiologyevents in Drive":  "/content/drive/MyDrive/microbiologyevents.csv.gz",
}
for name, path in checks.items():
    if os.path.exists(path):
        size = os.path.getsize(path)/1e6 if os.path.isfile(path) else None
        print("YES ", name, f"({size:.0f} MB)" if size is not None else "")
    else:
        print("NO  ", name)

In [ ]:
d = '/content/mimic/mimiciv/Hosp'
for t in ['prescriptions', 'microbiologyevents']:
    !mkdir -p "{d}/{t}.csv"
    !cp "/content/drive/MyDrive/{t}.csv.gz" "{d}/{t}.csv/{t}.csv.gz"
!ls -lh {d}/prescriptions.csv {d}/microbiologyevents.csv

In [ ]:
!pip install -q duckdb
import duckdb
con = duckdb.connect()
base = '/content/mimic/mimiciv'
P = lambda folder, name: f"{base}/{folder}/{name}.csv/{name}.csv.gz"

con.execute(f"CREATE TABLE patients   AS SELECT * FROM read_csv('{P('Hosp','patients')}', header=true)")
con.execute(f"CREATE TABLE admissions AS SELECT * FROM read_csv('{P('Hosp','admissions')}', header=true)")
con.execute(f"CREATE TABLE icustays   AS SELECT * FROM read_csv('{P('ICU','icustays')}', header=true)")

con.execute(f"""
CREATE TABLE abx AS
SELECT hadm_id, starttime AS abx_time, drug, route
FROM read_csv('{P('Hosp','prescriptions')}', header=true,
              types={{'starttime':'TIMESTAMP','stoptime':'TIMESTAMP'}}, ignore_errors=true)
WHERE hadm_id IS NOT NULL AND starttime IS NOT NULL
  AND regexp_matches(lower(drug),
      'cillin|cef|penem|floxacin|mycin|cycline|vancomycin|azithro|metronidazole|sulfameth|trimethoprim|linezolid|daptomycin|aztreonam|colistin|polymyxin|gentamicin|tobramycin|amikacin|tazobactam|clavulan|sulbactam|rifampin|tigecycline')
  AND NOT regexp_matches(lower(drug), 'bleomycin|mitomycin|dactinomycin|rapamycin')
  AND coalesce(route,'') NOT IN ('OU','OS','OD','AU','AS','AD','TP','IH')
""")

con.execute(f"""
CREATE TABLE cx AS
SELECT DISTINCT hadm_id, micro_specimen_id,
       coalesce(charttime, CAST(chartdate AS TIMESTAMP)) AS cx_time
FROM read_csv('{P('Hosp','microbiologyevents')}', header=true,
              types={{'charttime':'TIMESTAMP','chartdate':'DATE'}}, ignore_errors=true)
WHERE hadm_id IS NOT NULL
""")

con.execute("""
CREATE TABLE suspected AS
SELECT hadm_id, MIN(si_time) AS si_time FROM (
  SELECT a.hadm_id,
    CASE WHEN a.abx_time <= c.cx_time AND c.cx_time <= a.abx_time + INTERVAL '72 hours' THEN a.abx_time
         WHEN c.cx_time  <  a.abx_time AND a.abx_time <= c.cx_time + INTERVAL '24 hours' THEN c.cx_time END AS si_time
  FROM abx a JOIN cx c USING (hadm_id)
) WHERE si_time IS NOT NULL GROUP BY hadm_id
""")

con.execute("""
CREATE TABLE cohort_mimic AS
SELECT i.subject_id, i.hadm_id, i.stay_id, i.intime, i.outtime, i.los, s.si_time,
       p.gender, a.race, a.hospital_expire_flag,
       p.anchor_age + (year(a.admittime) - p.anchor_year) AS age
FROM icustays i
JOIN suspected s  USING (hadm_id)
JOIN admissions a USING (hadm_id)
JOIN patients p   ON p.subject_id = i.subject_id
WHERE s.si_time BETWEEN i.intime - INTERVAL '48 hours' AND i.intime + INTERVAL '24 hours'
QUALIFY row_number() OVER (PARTITION BY i.hadm_id ORDER BY i.intime) = 1
""")
con.execute("DELETE FROM cohort_mimic WHERE age < 18")

print(con.execute("""
SELECT (SELECT COUNT(*) FROM abx)               AS antibiotic_orders,
       (SELECT COUNT(DISTINCT hadm_id) FROM cx) AS admissions_with_cultures,
       (SELECT COUNT(*) FROM suspected)         AS suspected_infection_adm,
       (SELECT COUNT(*) FROM cohort_mimic)      AS icu_cohort_stays
""").df())

In [ ]:
!ls /content/mimic
!echo "--- inside mimiciv:"
!ls /content/mimic/mimiciv
!echo "--- inside Hosp:"
!ls /content/mimic/mimiciv/Hosp
!echo "--- where is patients.csv.gz:"
!find /content -name "patients.csv.gz" 2>/dev/null

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob

# 1. copy zip to local disk and unzip into a temp folder
!cp "/content/drive/MyDrive/MIMIC IV.zip" /content/mimic_iv.zip
!unzip -q /content/mimic_iv.zip -d /content/mimic_tmp
!rm /content/mimic_iv.zip

# 2. move everything into /content/mimic/mimiciv, keeping prescriptions and microbiologyevents
top = [d for d in os.listdir('/content/mimic_tmp')][0]
src_root = f'/content/mimic_tmp/{top}'
dst_root = '/content/mimic/mimiciv'
for sub in os.listdir(src_root):
    os.makedirs(f'{dst_root}/{sub}', exist_ok=True)
    for item in os.listdir(f'{src_root}/{sub}'):
        dst = f'{dst_root}/{sub}/{item}'
        if not os.path.exists(dst):
            shutil.move(f'{src_root}/{sub}/{item}', dst)
shutil.rmtree('/content/mimic_tmp')

# 3. delete uncompressed duplicates to save disk
for gz in glob.glob(f'{dst_root}/*/*/*.csv.gz'):
    plain = gz[:-3]
    if os.path.exists(plain):
        os.remove(plain)

print(sorted(os.path.basename(f) for f in glob.glob(f'{dst_root}/*/*/*.csv.gz')))
!df -h /content

In [ ]:
!pip install -q duckdb
import duckdb
con = duckdb.connect()
base = '/content/mimic/mimiciv'
P = lambda folder, name: f"{base}/{folder}/{name}.csv/{name}.csv.gz"

con.execute(f"CREATE TABLE patients   AS SELECT * FROM read_csv('{P('Hosp','patients')}', header=true)")
con.execute(f"CREATE TABLE admissions AS SELECT * FROM read_csv('{P('Hosp','admissions')}', header=true)")
con.execute(f"CREATE TABLE icustays   AS SELECT * FROM read_csv('{P('ICU','icustays')}', header=true)")

con.execute(f"""
CREATE TABLE abx AS
SELECT hadm_id, starttime AS abx_time, drug, route
FROM read_csv('{P('Hosp','prescriptions')}', header=true,
              types={{'starttime':'TIMESTAMP','stoptime':'TIMESTAMP'}}, ignore_errors=true)
WHERE hadm_id IS NOT NULL AND starttime IS NOT NULL
  AND regexp_matches(lower(drug),
      'cillin|cef|penem|floxacin|mycin|cycline|vancomycin|azithro|metronidazole|sulfameth|trimethoprim|linezolid|daptomycin|aztreonam|colistin|polymyxin|gentamicin|tobramycin|amikacin|tazobactam|clavulan|sulbactam|rifampin|tigecycline')
  AND NOT regexp_matches(lower(drug), 'bleomycin|mitomycin|dactinomycin|rapamycin')
  AND coalesce(route,'') NOT IN ('OU','OS','OD','AU','AS','AD','TP','IH')
""")

con.execute(f"""
CREATE TABLE cx AS
SELECT DISTINCT hadm_id, micro_specimen_id,
       coalesce(charttime, CAST(chartdate AS TIMESTAMP)) AS cx_time
FROM read_csv('{P('Hosp','microbiologyevents')}', header=true,
              types={{'charttime':'TIMESTAMP','chartdate':'DATE'}}, ignore_errors=true)
WHERE hadm_id IS NOT NULL
""")

con.execute("""
CREATE TABLE suspected AS
SELECT hadm_id, MIN(si_time) AS si_time FROM (
  SELECT a.hadm_id,
    CASE WHEN a.abx_time <= c.cx_time AND c.cx_time <= a.abx_time + INTERVAL '72 hours' THEN a.abx_time
         WHEN c.cx_time  <  a.abx_time AND a.abx_time <= c.cx_time + INTERVAL '24 hours' THEN c.cx_time END AS si_time
  FROM abx a JOIN cx c USING (hadm_id)
) WHERE si_time IS NOT NULL GROUP BY hadm_id
""")

con.execute("""
CREATE TABLE cohort_mimic AS
SELECT i.subject_id, i.hadm_id, i.stay_id, i.intime, i.outtime, i.los, s.si_time,
       p.gender, a.race, a.hospital_expire_flag,
       p.anchor_age + (year(a.admittime) - p.anchor_year) AS age
FROM icustays i
JOIN suspected s  USING (hadm_id)
JOIN admissions a USING (hadm_id)
JOIN patients p   ON p.subject_id = i.subject_id
WHERE s.si_time BETWEEN i.intime - INTERVAL '48 hours' AND i.intime + INTERVAL '24 hours'
QUALIFY row_number() OVER (PARTITION BY i.hadm_id ORDER BY i.intime) = 1
""")
con.execute("DELETE FROM cohort_mimic WHERE age < 18")

print(con.execute("""
SELECT (SELECT COUNT(*) FROM abx)               AS antibiotic_orders,
       (SELECT COUNT(DISTINCT hadm_id) FROM cx) AS admissions_with_cultures,
       (SELECT COUNT(*) FROM suspected)         AS suspected_infection_adm,
       (SELECT COUNT(*) FROM cohort_mimic)      AS icu_cohort_stays
""").df())

# save to Drive so a Colab reset doesn't lose it
os.makedirs('/content/drive/MyDrive/sepsis_project', exist_ok=True)
con.execute("COPY cohort_mimic TO '/content/drive/MyDrive/sepsis_project/cohort_mimic.parquet' (FORMAT PARQUET)")

In [ ]:
import duckdb, os
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
base = '/content/mimic/mimiciv'
P = lambda folder, name: f"{base}/{folder}/{name}.csv/{name}.csv.gz"
OUT = '/content/drive/MyDrive/sepsis_project'

con.execute(f"CREATE TABLE cohort AS SELECT * FROM read_parquet('{OUT}/cohort_mimic.parquet')")

# 1. chartevents: vitals, GCS, FiO2
chart_items = [220045, 220050, 220179, 220051, 220180, 220052, 220181, 225312,
               220210, 220277, 223762, 223761, 220739, 223900, 223901, 223835]
con.execute(f"""
COPY (
  SELECT c.stay_id, c.charttime, c.itemid, c.valuenum
  FROM read_csv('{P('ICU','chartevents')}', header=true,
       columns={{'subject_id':'BIGINT','hadm_id':'BIGINT','stay_id':'BIGINT','caregiver_id':'BIGINT',
                'charttime':'TIMESTAMP','storetime':'TIMESTAMP','itemid':'INTEGER',
                'value':'VARCHAR','valuenum':'DOUBLE','valueuom':'VARCHAR','warning':'INTEGER'}}) c
  JOIN cohort k ON k.stay_id = c.stay_id
  WHERE c.itemid IN ({','.join(map(str, chart_items))})
    AND c.charttime BETWEEN k.intime - INTERVAL '24 hours' AND k.outtime
) TO '{OUT}/chart_sub.parquet' (FORMAT PARQUET)
""")
print("chartevents done")

# 2. labevents: creatinine, bilirubin, platelets, lactate, WBC, PaO2, FiO2, etc.
lab_items = [50912, 50885, 51265, 50813, 51301, 51300, 50821, 50816, 50882, 50971, 50983, 51222, 50820, 50802]
con.execute(f"""
COPY (
  SELECT l.subject_id, l.hadm_id, l.charttime, l.itemid, l.valuenum
  FROM read_csv('{P('Hosp','labevents')}', header=true, ignore_errors=true,
       types={{'charttime':'TIMESTAMP','valuenum':'DOUBLE','itemid':'INTEGER'}}) l
  JOIN cohort k ON k.subject_id = l.subject_id
  WHERE l.itemid IN ({','.join(map(str, lab_items))})
    AND l.charttime BETWEEN k.intime - INTERVAL '24 hours' AND k.outtime
) TO '{OUT}/lab_sub.parquet' (FORMAT PARQUET)
""")
print("labevents done")

# 3. inputevents: vasopressors
vaso_items = [221906, 221289, 221749, 222315, 221662, 221653]
con.execute(f"""
COPY (
  SELECT i.stay_id, i.starttime, i.endtime, i.itemid, i.rate, i.rateuom, i.amount
  FROM read_csv('{P('ICU','inputevents')}', header=true, ignore_errors=true,
       types={{'starttime':'TIMESTAMP','endtime':'TIMESTAMP'}}) i
  JOIN cohort k ON k.stay_id = i.stay_id
  WHERE i.itemid IN ({','.join(map(str, vaso_items))})
) TO '{OUT}/vaso_sub.parquet' (FORMAT PARQUET)
""")
print("inputevents done")

for f in ['chart_sub', 'lab_sub', 'vaso_sub']:
    n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT}/{f}.parquet')").fetchone()[0]
    print(f, f"{n:,} rows")

In [ ]:
import duckdb, os
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
base = '/content/mimic/mimiciv'
P = lambda folder, name: f"{base}/{folder}/{name}.csv/{name}.csv.gz"
OUT = '/content/drive/MyDrive/sepsis_project'

con.execute(f"CREATE TABLE cohort AS SELECT * FROM read_parquet('{OUT}/cohort_mimic.parquet')")

# 1. chartevents: vitals, GCS, FiO2
chart_items = [220045, 220050, 220179, 220051, 220180, 220052, 220181, 225312,
               220210, 220277, 223762, 223761, 220739, 223900, 223901, 223835]
con.execute(f"""
COPY (
  SELECT c.stay_id, c.charttime, c.itemid, c.valuenum
  FROM read_csv('{P('ICU','chartevents')}', header=true, strict_mode=false, ignore_errors=true,
       columns={{'subject_id':'BIGINT','hadm_id':'BIGINT','stay_id':'BIGINT','caregiver_id':'BIGINT',
                'charttime':'TIMESTAMP','storetime':'TIMESTAMP','itemid':'BIGINT',
                'value':'VARCHAR','valuenum':'DOUBLE','valueuom':'VARCHAR','warning':'BIGINT'}}) c
  JOIN cohort k ON k.stay_id = c.stay_id
  WHERE c.itemid IN ({','.join(map(str, chart_items))})
    AND c.charttime BETWEEN k.intime - INTERVAL '24 hours' AND k.outtime
) TO '{OUT}/chart_sub.parquet' (FORMAT PARQUET)
""")
print("chartevents done")

# 2. labevents: creatinine, bilirubin, platelets, lactate, WBC, PaO2, etc.
lab_items = [50912, 50885, 51265, 50813, 51301, 51300, 50821, 50816, 50882, 50971, 50983, 51222, 50820, 50802]
con.execute(f"""
COPY (
  SELECT l.subject_id, l.hadm_id, l.charttime, l.itemid, l.valuenum
  FROM read_csv('{P('Hosp','labevents')}', header=true, strict_mode=false, ignore_errors=true,
       types={{'charttime':'TIMESTAMP','valuenum':'DOUBLE','itemid':'BIGINT'}}) l
  JOIN cohort k ON k.subject_id = l.subject_id
  WHERE l.itemid IN ({','.join(map(str, lab_items))})
    AND l.charttime BETWEEN k.intime - INTERVAL '24 hours' AND k.outtime
) TO '{OUT}/lab_sub.parquet' (FORMAT PARQUET)
""")
print("labevents done")

# 3. inputevents: vasopressors
vaso_items = [221906, 221289, 221749, 222315, 221662, 221653]
con.execute(f"""
COPY (
  SELECT i.stay_id, i.starttime, i.endtime, i.itemid, i.rate, i.rateuom, i.amount
  FROM read_csv('{P('ICU','inputevents')}', header=true, strict_mode=false, ignore_errors=true,
       types={{'starttime':'TIMESTAMP','endtime':'TIMESTAMP','itemid':'BIGINT'}}) i
  JOIN cohort k ON k.stay_id = i.stay_id
  WHERE i.itemid IN ({','.join(map(str, vaso_items))})
) TO '{OUT}/vaso_sub.parquet' (FORMAT PARQUET)
""")
print("inputevents done")

for f in ['chart_sub', 'lab_sub', 'vaso_sub']:
    n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT}/{f}.parquet')").fetchone()[0]
    print(f, f"{n:,} rows")


In [ ]:
import duckdb
con = duckdb.connect()
base = '/content/mimic/mimiciv'
OUT = '/content/drive/MyDrive/sepsis_project'

items = con.execute(f"""
  SELECT itemid, label FROM read_csv('{base}/ICU/d_items.csv/d_items.csv.gz', header=true)
""").df()
labitems = con.execute(f"""
  SELECT itemid, label, fluid FROM read_csv('{base}/Hosp/d_labitems.csv/d_labitems.csv.gz', header=true)
""").df()
con.register('items', items); con.register('labitems', labitems)

print("=== CHART items ===")
print(con.execute(f"""
  SELECT c.itemid, i.label, COUNT(*) AS n
  FROM read_parquet('{OUT}/chart_sub.parquet') c LEFT JOIN items i USING (itemid)
  GROUP BY 1,2 ORDER BY n DESC
""").df().to_string(index=False))

print("\n=== LAB items ===")
print(con.execute(f"""
  SELECT l.itemid, d.label, d.fluid, COUNT(*) AS n
  FROM read_parquet('{OUT}/lab_sub.parquet') l LEFT JOIN labitems d USING (itemid)
  GROUP BY 1,2,3 ORDER BY n DESC
""").df().to_string(index=False))

print("\n=== VASOPRESSOR items ===")
print(con.execute(f"""
  SELECT v.itemid, i.label, COUNT(*) AS n
  FROM read_parquet('{OUT}/vaso_sub.parquet') v LEFT JOIN items i USING (itemid)
  GROUP BY 1,2 ORDER BY n DESC
""").df().to_string(index=False))

In [ ]:
import duckdb, os
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
OUT = '/content/drive/MyDrive/sepsis_project'
R = lambda f: f"read_parquet('{OUT}/{f}.parquet')"

con.execute(f"CREATE TABLE cohort AS SELECT * FROM {R('cohort_mimic')}")

# hourly grid: ICU admission -24h to +7 days (or discharge)
con.execute("""
CREATE TABLE grid AS
SELECT stay_id, unnest(generate_series(
    date_trunc('hour', intime) - INTERVAL '24 hours',
    LEAST(date_trunc('hour', outtime), date_trunc('hour', intime) + INTERVAL '168 hours'),
    INTERVAL '1 hour')) AS hr
FROM cohort
""")

# hourly chart values
con.execute(f"""
CREATE TABLE h_chart AS
SELECT stay_id, date_trunc('hour', charttime) AS hr,
  MIN(CASE WHEN itemid IN (220052,225312,220181) THEN valuenum END) AS map_min,
  MIN(CASE WHEN itemid=220739 THEN valuenum END) AS gcs_eye,
  MIN(CASE WHEN itemid=223900 THEN valuenum END) AS gcs_verbal,
  MIN(CASE WHEN itemid=223901 THEN valuenum END) AS gcs_motor,
  MAX(CASE WHEN itemid=223835 THEN CASE WHEN valuenum>1 THEN valuenum/100.0 ELSE valuenum END END) AS fio2_chart
FROM {R('chart_sub')} GROUP BY 1,2
""")

# hourly lab values
con.execute(f"""
CREATE TABLE h_lab AS
SELECT k.stay_id, date_trunc('hour', l.charttime) AS hr,
  MIN(CASE WHEN l.itemid=50821 THEN l.valuenum END) AS pao2,
  MAX(CASE WHEN l.itemid=50816 THEN l.valuenum END) AS fio2_lab,
  MIN(CASE WHEN l.itemid=51265 THEN l.valuenum END) AS plt,
  MAX(CASE WHEN l.itemid=50885 THEN l.valuenum END) AS bili,
  MAX(CASE WHEN l.itemid=50912 THEN l.valuenum END) AS creat,
  MAX(CASE WHEN l.itemid=50813 THEN l.valuenum END) AS lactate
FROM {R('lab_sub')} l
JOIN cohort k ON k.subject_id=l.subject_id
 AND l.charttime BETWEEN k.intime - INTERVAL '24 hours' AND k.outtime
GROUP BY 1,2
""")

# hourly vasopressors (expand each infusion into the hours it ran)
con.execute(f"""
CREATE TABLE h_vaso AS
SELECT g.stay_id, g.hr,
  MAX(CASE WHEN v.itemid IN (221906,221289) THEN coalesce(v.rate,1) END) AS ne_epi,
  MAX(CASE WHEN v.itemid=221662 THEN coalesce(v.rate,1) END) AS dopa,
  MAX(CASE WHEN v.itemid IN (222315,221749) THEN 1 END) AS other_p,
  MAX(CASE WHEN v.itemid=221653 THEN 1 END) AS dobu,
  MAX(CASE WHEN v.itemid IN (221906,221289,221662,222315,221749) THEN 1 END) AS any_pressor
FROM grid g
JOIN {R('vaso_sub')} v ON v.stay_id=g.stay_id
 AND v.starttime < g.hr + INTERVAL '1 hour' AND v.endtime > g.hr
GROUP BY 1,2
""")

# SOFA: worst value over the past 24 hours
con.execute("""
CREATE TABLE sofa AS
WITH a AS (
  SELECT g.stay_id, g.hr, c.map_min,
         c.gcs_eye + c.gcs_verbal + c.gcs_motor AS gcs,
         l.pao2, l.plt, l.bili, l.creat, l.lactate,
         coalesce(c.fio2_chart, l.fio2_lab/100.0) AS fio2,
         coalesce(v.ne_epi,0) AS ne_epi, coalesce(v.dopa,0) AS dopa,
         coalesce(v.other_p,0) AS other_p, coalesce(v.dobu,0) AS dobu,
         coalesce(v.any_pressor,0) AS any_pressor
  FROM grid g
  LEFT JOIN h_chart c USING (stay_id, hr)
  LEFT JOIN h_lab   l USING (stay_id, hr)
  LEFT JOIN h_vaso  v USING (stay_id, hr)
), b AS (
  SELECT *, pao2 / coalesce(last_value(fio2 IGNORE NULLS) OVER
        (PARTITION BY stay_id ORDER BY hr ROWS BETWEEN 4 PRECEDING AND CURRENT ROW), 0.21) AS pf
  FROM a
), w AS (
  SELECT stay_id, hr, lactate, any_pressor,
    MIN(pf)    OVER win AS pf_min,   MIN(plt)  OVER win AS plt_min,
    MAX(bili)  OVER win AS bili_max, MAX(creat) OVER win AS creat_max,
    MIN(map_min) OVER win AS map24,  MIN(gcs)  OVER win AS gcs_min,
    MAX(ne_epi) OVER win AS ne_epi,  MAX(dopa) OVER win AS dopa,
    MAX(other_p) OVER win AS other_p, MAX(dobu) OVER win AS dobu
  FROM b
  WINDOW win AS (PARTITION BY stay_id ORDER BY hr ROWS BETWEEN 23 PRECEDING AND CURRENT ROW)
), s AS (
  SELECT *,
    CASE WHEN pf_min<100 THEN 4 WHEN pf_min<200 THEN 3 WHEN pf_min<300 THEN 2 WHEN pf_min<400 THEN 1 ELSE 0 END AS s_resp,
    CASE WHEN plt_min<20 THEN 4 WHEN plt_min<50 THEN 3 WHEN plt_min<100 THEN 2 WHEN plt_min<150 THEN 1 ELSE 0 END AS s_coag,
    CASE WHEN bili_max>=12 THEN 4 WHEN bili_max>=6 THEN 3 WHEN bili_max>=2 THEN 2 WHEN bili_max>=1.2 THEN 1 ELSE 0 END AS s_liver,
    CASE WHEN ne_epi>0.1 OR dopa>15 THEN 4
         WHEN ne_epi>0 OR dopa>5 OR other_p=1 THEN 3
         WHEN dopa>0 OR dobu=1 THEN 2
         WHEN map24<70 THEN 1 ELSE 0 END AS s_cv,
    CASE WHEN gcs_min<6 THEN 4 WHEN gcs_min<=9 THEN 3 WHEN gcs_min<=12 THEN 2 WHEN gcs_min<=14 THEN 1 ELSE 0 END AS s_cns,
    CASE WHEN creat_max>=5 THEN 4 WHEN creat_max>=3.5 THEN 3 WHEN creat_max>=2 THEN 2 WHEN creat_max>=1.2 THEN 1 ELSE 0 END AS s_renal
  FROM w
)
SELECT *, s_resp+s_coag+s_liver+s_cv+s_cns+s_renal AS sofa FROM s
""")
con.execute(f"COPY sofa TO '{OUT}/sofa_hourly.parquet' (FORMAT PARQUET)")

# Sepsis-3: SOFA >= 2 within -48h/+24h of suspected infection time (baseline SOFA assumed 0)
con.execute("""
CREATE TABLE sepsis3 AS
SELECT k.*, MIN(s.hr) AS sofa2_time
FROM cohort k JOIN sofa s ON s.stay_id=k.stay_id
 AND s.hr BETWEEN date_trunc('hour', k.si_time) - INTERVAL '48 hours'
              AND date_trunc('hour', k.si_time) + INTERVAL '24 hours'
 AND s.sofa >= 2
GROUP BY ALL
""")
con.execute(f"COPY sepsis3 TO '{OUT}/sepsis3_mimic.parquet' (FORMAT PARQUET)")

print(con.execute("""
SELECT (SELECT COUNT(*) FROM cohort)  AS suspected_infection_stays,
       (SELECT COUNT(*) FROM sepsis3) AS sepsis3_stays,
       (SELECT COUNT(*) FROM sofa)    AS hourly_rows
""").df())
print(con.execute("SELECT sofa, COUNT(*) AS hours FROM sofa GROUP BY 1 ORDER BY 1").df().T)

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
OUT = '/content/drive/MyDrive/sepsis_project'
R = lambda f: f"read_parquet('{OUT}/{f}.parquet')"

con.execute(f"CREATE TABLE sofa AS SELECT stay_id, hr, lactate, any_pressor, sofa FROM {R('sofa_hourly')}")
con.execute(f"CREATE TABLE s3   AS SELECT * FROM {R('sepsis3_mimic')}")

# shock onset = first hour with a vasopressor running and lactate > 2 within +/- 6h
con.execute("""
CREATE TABLE shock AS
SELECT stay_id, MIN(hr) AS shock_time FROM (
  SELECT stay_id, hr, any_pressor,
         MAX(lactate) OVER (PARTITION BY stay_id ORDER BY hr
                            ROWS BETWEEN 6 PRECEDING AND 6 FOLLOWING) AS lac_win
  FROM sofa
) WHERE any_pressor = 1 AND lac_win > 2
GROUP BY stay_id
""")

con.execute("""
CREATE TABLE labels AS
SELECT s.*, k.shock_time,
       GREATEST(date_trunc('hour', s.si_time), date_trunc('hour', s.intime)) AS index_time
FROM s3 s LEFT JOIN shock k USING (stay_id)
""")

con.execute("""
CREATE TABLE cohort_final AS
SELECT *, CASE WHEN shock_time IS NOT NULL THEN 1 ELSE 0 END AS shock,
       date_diff('hour', index_time, shock_time) AS hours_to_shock
FROM labels
WHERE index_time < outtime
  AND (shock_time IS NULL OR shock_time > index_time)
""")
con.execute(f"COPY cohort_final TO '{OUT}/cohort_final_mimic.parquet' (FORMAT PARQUET)")

print(con.execute("""
SELECT (SELECT COUNT(*) FROM labels) AS sepsis3_stays,
       (SELECT COUNT(*) FROM labels WHERE shock_time <= index_time) AS shock_already_at_index,
       (SELECT COUNT(*) FROM cohort_final) AS final_cohort,
       (SELECT SUM(shock) FROM cohort_final) AS developed_shock,
       (SELECT ROUND(100.0*AVG(shock),1) FROM cohort_final) AS pct_shock,
       (SELECT COUNT(*) FROM cohort_final WHERE hours_to_shock <= 6)  AS shock_within_6h,
       (SELECT COUNT(*) FROM cohort_final WHERE hours_to_shock <= 24) AS shock_within_24h
""").df().T)

print(con.execute("""
SELECT shock, COUNT(*) AS n, ROUND(AVG(age),1) AS mean_age,
       ROUND(100*AVG(hospital_expire_flag),1) AS pct_died_in_hospital
FROM cohort_final GROUP BY shock ORDER BY shock
""").df())

In [ ]:
import duckdb, os
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
OUT = '/content/drive/MyDrive/sepsis_project'
base = '/content/mimic/mimiciv'
R = lambda f: f"read_parquet('{OUT}/{f}.parquet')"

con.execute(f"""
CREATE TABLE cf AS
SELECT c.*, p.anchor_year_group,
       c.index_time + INTERVAL '1 hour' AS pred_time
FROM {R('cohort_final_mimic')} c
JOIN read_csv('{base}/Hosp/patients.csv/patients.csv.gz', header=true) p USING (subject_id)
""")

# ---- vitals in the 24h before prediction time ----
con.execute(f"""
CREATE TABLE chart_long AS
SELECT cf.stay_id, ch.charttime,
  CASE WHEN ch.itemid=220045 THEN 'hr' WHEN ch.itemid=220210 THEN 'rr' WHEN ch.itemid=220277 THEN 'spo2'
       WHEN ch.itemid IN (220179,220050) THEN 'sbp' WHEN ch.itemid IN (220180,220051) THEN 'dbp'
       WHEN ch.itemid IN (220181,220052,225312) THEN 'map'
       WHEN ch.itemid IN (223762,223761) THEN 'temp' END AS var,
  CASE WHEN ch.itemid=223761 THEN (ch.valuenum-32)/1.8 ELSE ch.valuenum END AS v
FROM {R('chart_sub')} ch JOIN cf USING (stay_id)
WHERE ch.itemid IN (220045,220210,220277,220179,220050,220180,220051,220181,220052,225312,223762,223761)
  AND ch.charttime <  cf.pred_time
  AND ch.charttime >= cf.pred_time - INTERVAL '24 hours'
""")
cv = ['hr','rr','spo2','sbp','dbp','map','temp']
aggs = []
for x in cv:
    aggs += [f"AVG(v) FILTER (WHERE var='{x}') AS {x}_mean",
             f"MIN(v) FILTER (WHERE var='{x}') AS {x}_min",
             f"MAX(v) FILTER (WHERE var='{x}') AS {x}_max",
             f"arg_max(v, charttime) FILTER (WHERE var='{x}') AS {x}_last"]
con.execute(f"CREATE TABLE chart_feat AS SELECT stay_id, {', '.join(aggs)} FROM chart_long GROUP BY stay_id")

# ---- labs in the 24h before prediction time ----
labmap = {'creat':50912,'bili':50885,'plt':51265,'lactate':50813,'wbc':51301,'hgb':51222,
          'k':50971,'na':50983,'bicarb':50882,'ph':50820,'pao2':50821,'base_excess':50802}
case = "CASE " + " ".join(f"WHEN lb.itemid={i} THEN '{n}'" for n,i in labmap.items()) + " END"
con.execute(f"""
CREATE TABLE lab_long AS
SELECT cf.stay_id, lb.charttime, {case} AS var, lb.valuenum AS v
FROM {R('lab_sub')} lb JOIN cf ON cf.subject_id = lb.subject_id
WHERE lb.itemid IN ({','.join(map(str, labmap.values()))})
  AND lb.charttime <  cf.pred_time
  AND lb.charttime >= cf.pred_time - INTERVAL '24 hours'
""")
aggs = []
for x in labmap:
    aggs += [f"arg_max(v, charttime) FILTER (WHERE var='{x}') AS {x}_last",
             f"MIN(v) FILTER (WHERE var='{x}') AS {x}_min",
             f"MAX(v) FILTER (WHERE var='{x}') AS {x}_max"]
con.execute(f"CREATE TABLE lab_feat AS SELECT stay_id, {', '.join(aggs)} FROM lab_long GROUP BY stay_id")

# ---- SOFA state at the index hour ----
con.execute(f"""
CREATE TABLE sofa_idx AS
SELECT cf.stay_id, s.sofa AS sofa_total, s.s_resp, s.s_coag, s.s_liver, s.s_cv, s.s_cns, s.s_renal,
       s.gcs_min, s.pf_min, s.any_pressor AS on_pressor
FROM cf JOIN {R('sofa_hourly')} s ON s.stay_id = cf.stay_id AND s.hr = cf.index_time
""")

con.execute("""
CREATE TABLE features AS
SELECT cf.stay_id, cf.subject_id, cf.hadm_id, cf.anchor_year_group, cf.race, cf.gender,
       cf.age, (cf.gender='M')::INT AS male,
       date_diff('hour', cf.intime, cf.pred_time) AS hours_icu_to_pred,
       cf.shock, cf.hours_to_shock,
       ch.* EXCLUDE (stay_id), lb.* EXCLUDE (stay_id), so.* EXCLUDE (stay_id)
FROM cf
LEFT JOIN chart_feat ch USING (stay_id)
LEFT JOIN lab_feat   lb USING (stay_id)
LEFT JOIN sofa_idx   so USING (stay_id)
""")
con.execute(f"COPY features TO '{OUT}/features_mimic.parquet' (FORMAT PARQUET)")

df = con.execute("SELECT * FROM features").df()
print(df.shape, "| shock rate:", round(100*df.shock.mean(),1), "%")
miss = (df.isna().mean()*100).round(0).sort_values(ascending=False)
print("\nMost-missing features (% missing):")
print(miss.head(12).to_string())
print("\nStays by anchor_year_group:")
print(df.anchor_year_group.value_counts().sort_index().to_string())

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

OUT = '/content/drive/MyDrive/sepsis_project'
df = pd.read_parquet(f'{OUT}/features_mimic.parquet')

train_groups = ['2008 - 2010', '2011 - 2013', '2014 - 2016']
tr = df[df.anchor_year_group.isin(train_groups)]
te = df[~df.anchor_year_group.isin(train_groups)]

drop = ['stay_id','subject_id','hadm_id','anchor_year_group','race','gender','shock','hours_to_shock']
feats = [c for c in df.columns if c not in drop]
print(f"{len(feats)} features | train {len(tr):,} (shock {tr.shock.mean()*100:.1f}%) | test {len(te):,} (shock {te.shock.mean()*100:.1f}%)")

rf = make_pipeline(
    SimpleImputer(strategy='median', add_indicator=True),
    RandomForestClassifier(n_estimators=500, min_samples_leaf=5, class_weight='balanced_subsample',
                           n_jobs=-1, random_state=42))
rf.fit(tr[feats], tr.shock)
p_rf   = rf.predict_proba(te[feats])[:, 1]
p_sofa = te['sofa_total'].fillna(0).values.astype(float)   # SOFA-only benchmark
y = te.shock.values

def sens_at_spec(y, p, spec):
    fpr, tpr, _ = roc_curve(y, p)
    return tpr[np.searchsorted(fpr, 1 - spec, side='right') - 1]

def report(name, y, p, n_boot=200, seed=0):
    rng = np.random.default_rng(seed)
    stats = {'AUROC': [], 'AUPRC': [], 'Sens@80%spec': [], 'Sens@90%spec': []}
    idx_all = np.arange(len(y))
    for _ in range(n_boot):
        i = rng.choice(idx_all, len(idx_all))
        if y[i].sum() == 0: continue
        stats['AUROC'].append(roc_auc_score(y[i], p[i]))
        stats['AUPRC'].append(average_precision_score(y[i], p[i]))
        stats['Sens@80%spec'].append(sens_at_spec(y[i], p[i], 0.80))
        stats['Sens@90%spec'].append(sens_at_spec(y[i], p[i], 0.90))
    out = {'model': name}
    for k, v in stats.items():
        out[k] = f"{np.mean(v):.3f} ({np.percentile(v,2.5):.3f}-{np.percentile(v,97.5):.3f})"
    return out

res = pd.DataFrame([report('Random Forest', y, p_rf), report('SOFA only', y, p_sofa)])
print(res.to_string(index=False))
print(f"\nBaseline AUPRC for a random model = prevalence = {y.mean():.3f}")

# save predictions and the top features for later
te[['stay_id','shock','race','gender','age']].assign(p_rf=p_rf, p_sofa=p_sofa)\
  .to_parquet(f'{OUT}/preds_rf_mimic_test.parquet')
imp = pd.Series(rf[-1].feature_importances_,
                index=list(rf[0].get_feature_names_out(feats))).sort_values(ascending=False)
print("\nTop 15 features:")
print(imp.head(15).round(4).to_string())

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

OUT = '/content/drive/MyDrive/sepsis_project'
df = pd.read_parquet(f'{OUT}/features_mimic.parquet')

# ---- clean: find and remove infinite / absurd values ----
num = df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[num]).sum()
print("Columns with infinite values:\n", inf_counts[inf_counts > 0].to_string() or "none")
df[num] = df[num].replace([np.inf, -np.inf], np.nan)

# physiologically impossible values -> missing
limits = {'hr':(0,300),'rr':(0,100),'spo2':(0,100),'sbp':(0,350),'dbp':(0,250),'map':(0,300),'temp':(25,45),
          'pf':(0,800)}
for c in df.columns:
    for key,(lo,hi) in limits.items():
        if c.startswith(key + '_') or c == key:
            df.loc[(df[c] < lo) | (df[c] > hi), c] = np.nan
df.loc[df['pf_min'] > 800, 'pf_min'] = np.nan

train_groups = ['2008 - 2010', '2011 - 2013', '2014 - 2016']
tr = df[df.anchor_year_group.isin(train_groups)]
te = df[~df.anchor_year_group.isin(train_groups)]

drop = ['stay_id','subject_id','hadm_id','anchor_year_group','race','gender','shock','hours_to_shock']
feats = [c for c in df.columns if c not in drop]
print(f"{len(feats)} features | train {len(tr):,} (shock {tr.shock.mean()*100:.1f}%) | test {len(te):,} (shock {te.shock.mean()*100:.1f}%)")

rf = make_pipeline(
    SimpleImputer(strategy='median', add_indicator=True),
    RandomForestClassifier(n_estimators=500, min_samples_leaf=5, class_weight='balanced_subsample',
                           n_jobs=-1, random_state=42))
rf.fit(tr[feats], tr.shock)
p_rf   = rf.predict_proba(te[feats])[:, 1]
p_sofa = te['sofa_total'].fillna(0).values.astype(float)   # SOFA-only benchmark
y = te.shock.values

def sens_at_spec(y, p, spec):
    fpr, tpr, _ = roc_curve(y, p)
    return tpr[np.searchsorted(fpr, 1 - spec, side='right') - 1]

def report(name, y, p, n_boot=200, seed=0):
    rng = np.random.default_rng(seed)
    stats = {'AUROC': [], 'AUPRC': [], 'Sens@80%spec': [], 'Sens@90%spec': []}
    idx_all = np.arange(len(y))
    for _ in range(n_boot):
        i = rng.choice(idx_all, len(idx_all))
        if y[i].sum() == 0: continue
        stats['AUROC'].append(roc_auc_score(y[i], p[i]))
        stats['AUPRC'].append(average_precision_score(y[i], p[i]))
        stats['Sens@80%spec'].append(sens_at_spec(y[i], p[i], 0.80))
        stats['Sens@90%spec'].append(sens_at_spec(y[i], p[i], 0.90))
    out = {'model': name}
    for k, v in stats.items():
        out[k] = f"{np.mean(v):.3f} ({np.percentile(v,2.5):.3f}-{np.percentile(v,97.5):.3f})"
    return out

res = pd.DataFrame([report('Random Forest', y, p_rf), report('SOFA only', y, p_sofa)])
print(res.to_string(index=False))
print(f"\nBaseline AUPRC for a random model = prevalence = {y.mean():.3f}")

te[['stay_id','shock','race','gender','age']].assign(p_rf=p_rf, p_sofa=p_sofa)\
  .to_parquet(f'{OUT}/preds_rf_mimic_test.parquet')
imp = pd.Series(rf[-1].feature_importances_,
                index=list(rf[0].get_feature_names_out(feats))).sort_values(ascending=False)
print("\nTop 15 features:")
print(imp.head(15).round(4).to_string())

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

print("Shock rate by SOFA at index (test set):")
print(te.assign(sofa_bin=pd.cut(te.sofa_total, [-1,1,3,5,7,9,11,30]))
        .groupby('sofa_bin', observed=True).shock.agg(['mean','size']).round(3))
print("\nShock rate by on_pressor:", te.groupby('on_pressor').shock.mean().round(3).to_dict())

def fit_eval(name, cols, indicator):
    m = make_pipeline(SimpleImputer(strategy='median', add_indicator=indicator),
        RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                               class_weight='balanced_subsample', n_jobs=-1, random_state=42))
    m.fit(tr[cols], tr.shock)
    p = m.predict_proba(te[cols])[:, 1]
    print(f"{name:45s} AUROC {roc_auc_score(te.shock, p):.3f}  AUPRC {average_precision_score(te.shock, p):.3f}")

print()
fit_eval("All features (with missing flags)", feats, True)
fit_eval("No missing flags", feats, False)
fit_eval("No missing flags, no hours_icu_to_pred", [c for c in feats if c != 'hours_icu_to_pred'], False)

In [ ]:
import duckdb, pandas as pd
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
OUT = '/content/drive/MyDrive/sepsis_project'
base = '/content/mimic/mimiciv'
R = lambda f: f"read_parquet('{OUT}/{f}.parquet')"

# ---------- cohort with corrected index time ----------
con.execute(f"CREATE TABLE sofa AS SELECT stay_id, hr, lactate, any_pressor, sofa FROM {R('sofa_hourly')}")
con.execute("""
CREATE TABLE shock AS
SELECT stay_id, MIN(hr) AS shock_time FROM (
  SELECT stay_id, hr, any_pressor,
         MAX(lactate) OVER (PARTITION BY stay_id ORDER BY hr
                            ROWS BETWEEN 6 PRECEDING AND 6 FOLLOWING) AS lac_win
  FROM sofa
) WHERE any_pressor = 1 AND lac_win > 2
GROUP BY stay_id
""")
con.execute(f"""
CREATE TABLE cohort_final AS
SELECT *, CASE WHEN shock_time IS NOT NULL THEN 1 ELSE 0 END AS shock,
       date_diff('hour', index_time, shock_time) AS hours_to_shock
FROM (
  SELECT s.*, k.shock_time,
         GREATEST(date_trunc('hour', s.si_time), date_trunc('hour', s.sofa2_time),
                  date_trunc('hour', s.intime)) AS index_time
  FROM {R('sepsis3_mimic')} s LEFT JOIN shock k USING (stay_id)
)
WHERE index_time < outtime AND (shock_time IS NULL OR shock_time > index_time)
""")
con.execute(f"COPY cohort_final TO '{OUT}/cohort_final_mimic.parquet' (FORMAT PARQUET)")

# ---------- features (same as before) ----------
con.execute(f"""
CREATE TABLE cf AS
SELECT c.*, p.anchor_year_group, c.index_time + INTERVAL '1 hour' AS pred_time
FROM cohort_final c
JOIN read_csv('{base}/Hosp/patients.csv/patients.csv.gz', header=true) p USING (subject_id)
""")
con.execute(f"""
CREATE TABLE chart_long AS
SELECT cf.stay_id, ch.charttime,
  CASE WHEN ch.itemid=220045 THEN 'hr' WHEN ch.itemid=220210 THEN 'rr' WHEN ch.itemid=220277 THEN 'spo2'
       WHEN ch.itemid IN (220179,220050) THEN 'sbp' WHEN ch.itemid IN (220180,220051) THEN 'dbp'
       WHEN ch.itemid IN (220181,220052,225312) THEN 'map'
       WHEN ch.itemid IN (223762,223761) THEN 'temp' END AS var,
  CASE WHEN ch.itemid=223761 THEN (ch.valuenum-32)/1.8 ELSE ch.valuenum END AS v
FROM {R('chart_sub')} ch JOIN cf USING (stay_id)
WHERE ch.itemid IN (220045,220210,220277,220179,220050,220180,220051,220181,220052,225312,223762,223761)
  AND ch.charttime <  cf.pred_time AND ch.charttime >= cf.pred_time - INTERVAL '24 hours'
""")
aggs = []
for x in ['hr','rr','spo2','sbp','dbp','map','temp']:
    aggs += [f"AVG(v) FILTER (WHERE var='{x}') AS {x}_mean", f"MIN(v) FILTER (WHERE var='{x}') AS {x}_min",
             f"MAX(v) FILTER (WHERE var='{x}') AS {x}_max", f"arg_max(v, charttime) FILTER (WHERE var='{x}') AS {x}_last"]
con.execute(f"CREATE TABLE chart_feat AS SELECT stay_id, {', '.join(aggs)} FROM chart_long GROUP BY stay_id")

labmap = {'creat':50912,'bili':50885,'plt':51265,'lactate':50813,'wbc':51301,'hgb':51222,
          'k':50971,'na':50983,'bicarb':50882,'ph':50820,'pao2':50821,'base_excess':50802}
case = "CASE " + " ".join(f"WHEN lb.itemid={i} THEN '{n}'" for n,i in labmap.items()) + " END"
con.execute(f"""
CREATE TABLE lab_long AS
SELECT cf.stay_id, lb.charttime, {case} AS var, lb.valuenum AS v
FROM {R('lab_sub')} lb JOIN cf ON cf.subject_id = lb.subject_id
WHERE lb.itemid IN ({','.join(map(str, labmap.values()))})
  AND lb.charttime < cf.pred_time AND lb.charttime >= cf.pred_time - INTERVAL '24 hours'
""")
aggs = []
for x in labmap:
    aggs += [f"arg_max(v, charttime) FILTER (WHERE var='{x}') AS {x}_last",
             f"MIN(v) FILTER (WHERE var='{x}') AS {x}_min", f"MAX(v) FILTER (WHERE var='{x}') AS {x}_max"]
con.execute(f"CREATE TABLE lab_feat AS SELECT stay_id, {', '.join(aggs)} FROM lab_long GROUP BY stay_id")

con.execute(f"""
CREATE TABLE sofa_idx AS
SELECT cf.stay_id, s.sofa AS sofa_total, s.s_resp, s.s_coag, s.s_liver, s.s_cv, s.s_cns, s.s_renal,
       s.gcs_min, s.pf_min, s.any_pressor AS on_pressor
FROM cf JOIN {R('sofa_hourly')} s ON s.stay_id = cf.stay_id AND s.hr = cf.index_time
""")
con.execute("""
CREATE TABLE features AS
SELECT cf.stay_id, cf.subject_id, cf.hadm_id, cf.anchor_year_group, cf.race, cf.gender,
       cf.age, (cf.gender='M')::INT AS male,
       date_diff('hour', cf.intime, cf.pred_time) AS hours_icu_to_pred,
       cf.shock, cf.hours_to_shock,
       ch.* EXCLUDE (stay_id), lb.* EXCLUDE (stay_id), so.* EXCLUDE (stay_id)
FROM cf LEFT JOIN chart_feat ch USING (stay_id) LEFT JOIN lab_feat lb USING (stay_id) LEFT JOIN sofa_idx so USING (stay_id)
""")
con.execute(f"COPY features TO '{OUT}/features_mimic.parquet' (FORMAT PARQUET)")

df = con.execute("SELECT * FROM features").df()
print(df.shape, "| shock rate:", round(100*df.shock.mean(), 1), "%")
print("Shock within 6h:", int(((df.hours_to_shock <= 6) & (df.shock == 1)).sum()),
      "| within 24h:", int(((df.hours_to_shock <= 24) & (df.shock == 1)).sum()))
print("\nShock rate by SOFA at index:")
print(df.assign(sofa_bin=pd.cut(df.sofa_total, [-1,1,3,5,7,9,11,30]))
        .groupby('sofa_bin', observed=True).shock.agg(['mean','size']).round(3))
print("\nStays by anchor_year_group:")
print(df.anchor_year_group.value_counts().sort_index().to_string())

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

OUT = '/content/drive/MyDrive/sepsis_project'
df = pd.read_parquet(f'{OUT}/features_mimic.parquet')

num = df.select_dtypes(include=[np.number]).columns
df[num] = df[num].replace([np.inf, -np.inf], np.nan)
limits = {'hr':(0,300),'rr':(0,100),'spo2':(0,100),'sbp':(0,350),'dbp':(0,250),'map':(0,300),'temp':(25,45)}
for c in df.columns:
    for key,(lo,hi) in limits.items():
        if c.startswith(key + '_'):
            df.loc[(df[c] < lo) | (df[c] > hi), c] = np.nan
df.loc[df['pf_min'] > 800, 'pf_min'] = np.nan

train_groups = ['2008 - 2010', '2011 - 2013', '2014 - 2016']
tr = df[df.anchor_year_group.isin(train_groups)]
te = df[~df.anchor_year_group.isin(train_groups)]
drop = ['stay_id','subject_id','hadm_id','anchor_year_group','race','gender','shock','hours_to_shock']
feats = [c for c in df.columns if c not in drop]
print(f"{len(feats)} features | train {len(tr):,} (shock {tr.shock.mean()*100:.1f}%) | test {len(te):,} (shock {te.shock.mean()*100:.1f}%)")

rf = make_pipeline(
    SimpleImputer(strategy='median', add_indicator=True),
    RandomForestClassifier(n_estimators=500, min_samples_leaf=5, class_weight='balanced_subsample',
                           n_jobs=-1, random_state=42))
rf.fit(tr[feats], tr.shock)
p_rf   = rf.predict_proba(te[feats])[:, 1]
p_sofa = te['sofa_total'].fillna(0).values.astype(float)
y = te.shock.values

def sens_at_spec(y, p, spec):
    fpr, tpr, _ = roc_curve(y, p)
    return tpr[np.searchsorted(fpr, 1 - spec, side='right') - 1]

def report(name, y, p, n_boot=200, seed=0):
    rng = np.random.default_rng(seed)
    stats = {'AUROC': [], 'AUPRC': [], 'Sens@80%spec': [], 'Sens@90%spec': []}
    for _ in range(n_boot):
        i = rng.choice(len(y), len(y))
        if y[i].sum() == 0: continue
        stats['AUROC'].append(roc_auc_score(y[i], p[i]))
        stats['AUPRC'].append(average_precision_score(y[i], p[i]))
        stats['Sens@80%spec'].append(sens_at_spec(y[i], p[i], 0.80))
        stats['Sens@90%spec'].append(sens_at_spec(y[i], p[i], 0.90))
    out = {'model': name}
    for k, v in stats.items():
        out[k] = f"{np.mean(v):.3f} ({np.percentile(v,2.5):.3f}-{np.percentile(v,97.5):.3f})"
    return out

res = pd.DataFrame([report('Random Forest', y, p_rf), report('SOFA only', y, p_sofa)])
print(res.to_string(index=False))
print(f"\nBaseline AUPRC for a random model = prevalence = {y.mean():.3f}")

te[['stay_id','shock','race','gender','age']].assign(p_rf=p_rf, p_sofa=p_sofa)\
  .to_parquet(f'{OUT}/preds_rf_mimic_test.parquet')
imp = pd.Series(rf[-1].feature_importances_,
                index=list(rf[0].get_feature_names_out(feats))).sort_values(ascending=False)
print("\nTop 15 features:")
print(imp.head(15).round(4).to_string())

In [ ]:
!pip install -q shap
import shap, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt

OUT = '/content/drive/MyDrive/sepsis_project'
joblib.dump({'model': rf, 'feats': feats}, f'{OUT}/rf_mimic_model.joblib')

imputer, forest = rf[0], rf[-1]
names = list(imputer.get_feature_names_out(feats))

sample = te[feats].sample(1500, random_state=42)
X = pd.DataFrame(imputer.transform(sample), columns=names)

explainer = shap.TreeExplainer(forest)
sv = explainer.shap_values(X)
sv = sv[1] if isinstance(sv, list) else (sv[:, :, 1] if sv.ndim == 3 else sv)

plt.figure()
shap.summary_plot(sv, X, plot_type='bar', max_display=15, show=False)
plt.tight_layout(); plt.savefig(f'{OUT}/shap_bar.png', dpi=200); plt.show()

plt.figure()
shap.summary_plot(sv, X, max_display=15, show=False)
plt.tight_layout(); plt.savefig(f'{OUT}/shap_beeswarm.png', dpi=200); plt.show()

top = pd.Series(np.abs(sv).mean(0), index=names).sort_values(ascending=False).head(10)
print(top.round(4).to_string())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, subprocess

!apt-get -qq install unrar
!cp "/content/drive/MyDrive/eicu-collaborative-research-database-2.0.rar" /content/eicu.rar
!unrar x -idq /content/eicu.rar /content/eicu/
!rm /content/eicu.rar

SRC = '/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0'
Z   = '/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0.zip'
!unzip -j -n -q "{Z}" "*/patient.csv.gz" "*/vitalPeriodic.csv.gz" "*/vitalAperiodic.csv.gz" "*/nurseCharting.csv.gz" -d "{SRC}"

# verify the files we need against the official checksums
need = ['patient','vitalPeriodic','vitalAperiodic','nurseCharting','lab','infusionDrug','medication','microLab','apacheApsVar']
pattern = '|'.join(f' {n}\\.csv\\.gz' for n in need)
r = subprocess.run(f'cd "{SRC}" && grep -E "{pattern}" SHA256SUMS.txt | sha256sum -c', shell=True, capture_output=True, text=True)
print(r.stdout, r.stderr)
!ls -lh "{SRC}" | grep -E "patient|vital|nurseCharting|lab|infusion|medication|microLab|apacheAps"
!df -h /content

In [ ]:
!pip install -q duckdb
import duckdb, os
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
SRC = '/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0'
OUT = '/content/drive/MyDrive/sepsis_project'
F = lambda n: f"{SRC}/{n}.csv.gz"
opts = "header=true, strict_mode=false, ignore_errors=true"

# adults, first ICU stay of each hospital stay
con.execute(f"""
CREATE TABLE pat AS
SELECT patientunitstayid AS stay_id, patienthealthsystemstayid AS hadm_id, hospitalid,
       gender, ethnicity AS race, hospitaldischargeyear AS dis_year,
       unitdischargeoffset AS outoffset, hospitaldischargestatus, admissionweight,
       CASE WHEN age LIKE '>%' THEN 90 ELSE TRY_CAST(age AS DOUBLE) END AS age_num,
       unitvisitnumber
FROM read_csv('{F('patient')}', {opts}, types={{'age':'VARCHAR'}})
""")
con.execute("CREATE TABLE pat_adult AS SELECT * FROM pat WHERE age_num >= 18 AND unitvisitnumber = 1")

# antibiotics (same drug list as MIMIC), not cancelled, systemic routes
con.execute(f"""
CREATE TABLE abx AS
SELECT patientunitstayid AS stay_id, drugstartoffset AS abx_t, drugname
FROM read_csv('{F('medication')}', {opts})
WHERE drugstartoffset IS NOT NULL AND coalesce(drugordercancelled,'No') <> 'Yes'
  AND regexp_matches(lower(drugname),
      'cillin|cef|penem|floxacin|mycin|cycline|vancomycin|azithro|metronidazole|sulfameth|trimethoprim|linezolid|daptomycin|aztreonam|colistin|polymyxin|gentamicin|tobramycin|amikacin|tazobactam|clavulan|sulbactam|rifampin|tigecycline')
  AND NOT regexp_matches(lower(drugname), 'bleomycin|mitomycin|dactinomycin|rapamycin')
  AND NOT regexp_matches(lower(coalesce(routeadmin,'')), 'ophth|opht|topical|inhal|otic|ear|eye')
""")

con.execute(f"""
CREATE TABLE cx AS
SELECT DISTINCT patientunitstayid AS stay_id, culturetakenoffset AS cx_t
FROM read_csv('{F('microLab')}', {opts}) WHERE culturetakenoffset IS NOT NULL
""")

# suspected infection time in minutes from ICU admission (same rule as MIMIC: 72h / 24h)
con.execute("""
CREATE TABLE suspected AS
SELECT stay_id, MIN(si_t) AS si_t FROM (
  SELECT a.stay_id,
    CASE WHEN a.abx_t <= c.cx_t AND c.cx_t <= a.abx_t + 72*60 THEN a.abx_t
         WHEN c.cx_t  <  a.abx_t AND a.abx_t <= c.cx_t + 24*60 THEN c.cx_t END AS si_t
  FROM abx a JOIN cx c USING (stay_id)
) WHERE si_t IS NOT NULL GROUP BY stay_id
""")

con.execute("""
CREATE TABLE cohort_eicu AS
SELECT p.*, s.si_t
FROM pat_adult p JOIN suspected s USING (stay_id)
WHERE s.si_t BETWEEN -48*60 AND 24*60
""")
con.execute(f"COPY cohort_eicu TO '{OUT}/cohort_eicu.parquet' (FORMAT PARQUET)")

print(con.execute("""
SELECT (SELECT COUNT(*) FROM pat)                                   AS all_icu_stays,
       (SELECT COUNT(*) FROM pat_adult)                             AS adult_first_stays,
       (SELECT COUNT(DISTINCT stay_id) FROM abx)                    AS stays_with_antibiotic,
       (SELECT COUNT(DISTINCT stay_id) FROM cx)                     AS stays_with_culture,
       (SELECT COUNT(*) FROM suspected)                             AS suspected_infection,
       (SELECT COUNT(*) FROM cohort_eicu)                           AS eicu_cohort_stays,
       (SELECT COUNT(DISTINCT hospitalid) FROM cohort_eicu)         AS hospitals_in_cohort
""").df().T)

In [ ]:
!pip install -q duckdb
import duckdb
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
SRC = '/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0'
OUT = '/content/drive/MyDrive/sepsis_project'
F = lambda n: f"{SRC}/{n}.csv.gz"
opts = "header=true, strict_mode=false, ignore_errors=true"

con.execute(f"""
CREATE TABLE pat AS
SELECT patientunitstayid AS stay_id, patienthealthsystemstayid AS hadm_id, hospitalid,
       gender, ethnicity AS race, hospitaldischargeyear AS dis_year,
       unitdischargeoffset AS outoffset, hospitaldischargestatus, admissionweight,
       CASE WHEN age LIKE '>%' THEN 90 ELSE TRY_CAST(age AS DOUBLE) END AS age_num,
       unitvisitnumber
FROM read_csv('{F('patient')}', {opts}, types={{'age':'VARCHAR'}})
""")
con.execute("CREATE TABLE pat_adult AS SELECT * FROM pat WHERE age_num >= 18 AND unitvisitnumber = 1")

# antibiotics started within [-48h, +24h] of ICU admission
con.execute(f"""
CREATE TABLE abx AS
SELECT patientunitstayid AS stay_id, drugstartoffset AS abx_t
FROM read_csv('{F('medication')}', {opts})
WHERE drugstartoffset BETWEEN -48*60 AND 24*60 AND coalesce(drugordercancelled,'No') <> 'Yes'
  AND regexp_matches(lower(drugname),
      'cillin|cef|penem|floxacin|mycin|cycline|vancomycin|azithro|metronidazole|sulfameth|trimethoprim|linezolid|daptomycin|aztreonam|colistin|polymyxin|gentamicin|tobramycin|amikacin|tazobactam|clavulan|sulbactam|rifampin|tigecycline')
  AND NOT regexp_matches(lower(drugname), 'bleomycin|mitomycin|dactinomycin|rapamycin')
  AND NOT regexp_matches(lower(coalesce(routeadmin,'')), 'ophth|opht|topical|inhal|otic|ear|eye')
""")

con.execute(f"""
CREATE TABLE cx AS
SELECT DISTINCT patientunitstayid AS stay_id
FROM read_csv('{F('microLab')}', {opts})
WHERE culturetakenoffset BETWEEN -48*60 AND 24*60
""")

# sepsis diagnosis (shock excluded) entered by hour 24
con.execute(f"""
CREATE TABLE sepsis_dx AS
SELECT DISTINCT patientunitstayid AS stay_id
FROM read_csv('{F('diagnosis')}', {opts}, types={{'icd9code':'VARCHAR'}})
WHERE diagnosisoffset <= 24*60 AND (
     (lower(diagnosisstring) LIKE '%sepsis%' AND lower(diagnosisstring) NOT LIKE '%shock%')
  OR regexp_matches(coalesce(icd9code,''), '(^|, )(038|995\\.91)')
  OR regexp_matches(coalesce(icd9code,''), 'A40|A41'))
""")

con.execute("""
CREATE TABLE cohort_eicu AS
SELECT p.*, a.si_t
FROM pat_adult p
JOIN (SELECT stay_id, MIN(abx_t) AS si_t FROM abx GROUP BY stay_id) a USING (stay_id)
WHERE p.stay_id IN (SELECT stay_id FROM cx) OR p.stay_id IN (SELECT stay_id FROM sepsis_dx)
""")
con.execute(f"COPY cohort_eicu TO '{OUT}/cohort_eicu.parquet' (FORMAT PARQUET)")

print(con.execute("""
SELECT (SELECT COUNT(DISTINCT stay_id) FROM abx WHERE stay_id IN (SELECT stay_id FROM pat_adult)) AS adult_with_antibiotic_in_window,
       (SELECT COUNT(*) FROM cx   WHERE stay_id IN (SELECT stay_id FROM pat_adult)) AS with_culture,
       (SELECT COUNT(*) FROM sepsis_dx WHERE stay_id IN (SELECT stay_id FROM pat_adult)) AS with_sepsis_dx,
       (SELECT COUNT(*) FROM cohort_eicu)                    AS eicu_cohort_stays,
       (SELECT COUNT(DISTINCT hospitalid) FROM cohort_eicu)  AS hospitals_in_cohort
""").df().T)

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
SRC = '/content/eicu/eicu-collaborative-research-database-2.0/eicu-collaborative-research-database-2.0'
OUT = '/content/drive/MyDrive/sepsis_project'
F = lambda n: f"{SRC}/{n}.csv.gz"
opts = "header=true, strict_mode=false, ignore_errors=true"

con.execute(f"CREATE TABLE cohort AS SELECT stay_id, outoffset FROM read_parquet('{OUT}/cohort_eicu.parquet')")

# 1. vitals (periodic)
con.execute(f"""
COPY (
  SELECT v.patientunitstayid AS stay_id, v.observationoffset AS t,
         v.heartrate, v.respiration, v.sao2, v.temperature,
         v.systemicsystolic, v.systemicdiastolic, v.systemicmean
  FROM read_csv('{F('vitalPeriodic')}', {opts},
       types={{'observationoffset':'BIGINT','heartrate':'DOUBLE','respiration':'DOUBLE','sao2':'DOUBLE',
              'temperature':'DOUBLE','systemicsystolic':'DOUBLE','systemicdiastolic':'DOUBLE','systemicmean':'DOUBLE'}}) v
  JOIN cohort k ON k.stay_id = v.patientunitstayid
  WHERE v.observationoffset BETWEEN -1440 AND k.outoffset
) TO '{OUT}/eicu_vitalp_sub.parquet' (FORMAT PARQUET)
""")
print("vitalPeriodic done")

# 2. vitals (non-invasive blood pressure)
con.execute(f"""
COPY (
  SELECT v.patientunitstayid AS stay_id, v.observationoffset AS t,
         v.noninvasivesystolic, v.noninvasivediastolic, v.noninvasivemean
  FROM read_csv('{F('vitalAperiodic')}', {opts},
       types={{'observationoffset':'BIGINT','noninvasivesystolic':'DOUBLE','noninvasivediastolic':'DOUBLE','noninvasivemean':'DOUBLE'}}) v
  JOIN cohort k ON k.stay_id = v.patientunitstayid
  WHERE v.observationoffset BETWEEN -1440 AND k.outoffset
) TO '{OUT}/eicu_vitala_sub.parquet' (FORMAT PARQUET)
""")
print("vitalAperiodic done")

# 3. labs
labs = ['creatinine','total bilirubin','platelets x 1000','lactate','wbc x 1000','hgb','potassium',
        'sodium','bicarbonate','ph','pao2','base excess','fio2']
lab_list = ",".join(f"'{x}'" for x in labs)
con.execute(f"""
COPY (
  SELECT l.patientunitstayid AS stay_id, l.labresultoffset AS t, lower(l.labname) AS labname, l.labresult AS v
  FROM read_csv('{F('lab')}', {opts},
       types={{'labresultoffset':'BIGINT','labresult':'DOUBLE'}}) l
  JOIN cohort k ON k.stay_id = l.patientunitstayid
  WHERE lower(l.labname) IN ({lab_list})
    AND l.labresultoffset BETWEEN -1440 AND k.outoffset
) TO '{OUT}/eicu_lab_sub.parquet' (FORMAT PARQUET)
""")
print("lab done")

# 4. vasopressor infusions
con.execute(f"""
COPY (
  SELECT d.patientunitstayid AS stay_id, d.infusionoffset AS t, lower(d.drugname) AS drugname,
         TRY_CAST(d.drugrate AS DOUBLE) AS rate, TRY_CAST(d.patientweight AS DOUBLE) AS weight
  FROM read_csv('{F('infusionDrug')}', {opts},
       types={{'infusionoffset':'BIGINT','drugname':'VARCHAR','drugrate':'VARCHAR','patientweight':'VARCHAR'}}) d
  JOIN cohort k ON k.stay_id = d.patientunitstayid
  WHERE regexp_matches(lower(d.drugname), 'norepinephrine|epinephrine|dopamine|vasopressin|phenylephrine|dobutamine|levophed|neosynephrine')
    AND d.infusionoffset BETWEEN -1440 AND k.outoffset
) TO '{OUT}/eicu_infusion_sub.parquet' (FORMAT PARQUET)
""")
print("infusionDrug done")

# 5. Glasgow Coma Scale from nurse charting
con.execute(f"""
COPY (
  SELECT n.patientunitstayid AS stay_id, n.nursingchartoffset AS t,
         n.nursingchartcelltypevalname AS valname, TRY_CAST(n.nursingchartvalue AS DOUBLE) AS v
  FROM read_csv('{F('nurseCharting')}', {opts},
       types={{'nursingchartoffset':'BIGINT','nursingchartvalue':'VARCHAR'}}) n
  JOIN cohort k ON k.stay_id = n.patientunitstayid
  WHERE lower(n.nursingchartcelltypevallabel) LIKE 'glasgow%'
    AND n.nursingchartoffset BETWEEN -1440 AND k.outoffset
) TO '{OUT}/eicu_gcs_sub.parquet' (FORMAT PARQUET)
""")
print("nurseCharting done")

for f in ['eicu_vitalp_sub','eicu_vitala_sub','eicu_lab_sub','eicu_infusion_sub','eicu_gcs_sub']:
    n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUT}/{f}.parquet')").fetchone()[0]
    print(f, f"{n:,} rows")

print(con.execute(f"SELECT labname, COUNT(*) AS n FROM read_parquet('{OUT}/eicu_lab_sub.parquet') GROUP BY 1 ORDER BY n DESC").df().to_string(index=False))
print(con.execute(f"SELECT drugname, COUNT(*) AS n FROM read_parquet('{OUT}/eicu_infusion_sub.parquet') GROUP BY 1 ORDER BY n DESC LIMIT 15").df().to_string(index=False))
print(con.execute(f"SELECT valname, COUNT(*) AS n FROM read_parquet('{OUT}/eicu_gcs_sub.parquet') GROUP BY 1 ORDER BY n DESC").df().to_string(index=False))

In [ ]:
import duckdb, pandas as pd
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
OUT = '/content/drive/MyDrive/sepsis_project'
R = lambda f: f"read_parquet('{OUT}/{f}.parquet')"
H = lambda c: f"CAST(floor({c}/60.0) AS INTEGER)"      # offset in minutes -> hour index (0 = ICU admission)

con.execute(f"CREATE TABLE cohort AS SELECT * FROM {R('cohort_eicu')}")
con.execute(f"""CREATE TABLE grid AS
SELECT stay_id, unnest(generate_series(-24, LEAST({H('outoffset')}, 168))) AS hr FROM cohort""")

# ---------- vitals (invasive + non-invasive pooled, as in MIMIC) ----------
con.execute(f"""
CREATE TABLE vit AS
SELECT stay_id, t, heartrate AS hrt, respiration AS rrt, sao2 AS spo2t, temperature AS tempt,
       systemicsystolic AS sbpt, systemicdiastolic AS dbpt, systemicmean AS mapt FROM {R('eicu_vitalp_sub')}
UNION ALL
SELECT stay_id, t, NULL, NULL, NULL, NULL, noninvasivesystolic, noninvasivediastolic, noninvasivemean
FROM {R('eicu_vitala_sub')}
""")
con.execute(f"CREATE TABLE h_map AS SELECT stay_id, {H('t')} AS hr, MIN(mapt) AS map_min FROM vit GROUP BY 1,2")

# ---------- labs ----------
con.execute(f"CREATE TABLE lab AS SELECT * FROM {R('eicu_lab_sub')}")
con.execute(f"""
CREATE TABLE h_lab AS
SELECT stay_id, {H('t')} AS hr,
 MIN(CASE WHEN labname='pao2' THEN v END) AS pao2,
 MAX(CASE WHEN labname='fio2' THEN CASE WHEN v>1 THEN v/100.0 ELSE NULLIF(v,0) END END) AS fio2_lab,
 MIN(CASE WHEN labname='platelets x 1000' THEN v END) AS plt,
 MAX(CASE WHEN labname='total bilirubin' THEN v END) AS bili,
 MAX(CASE WHEN labname='creatinine' THEN v END) AS creat,
 MAX(CASE WHEN labname='lactate' THEN v END) AS lactate
FROM lab GROUP BY 1,2
""")

# ---------- GCS ----------
con.execute(f"""
CREATE TABLE h_gcs AS
SELECT stay_id, {H('t')} AS hr, MIN(v) AS gcs
FROM {R('eicu_gcs_sub')} WHERE valname='GCS Total' AND v BETWEEN 3 AND 15 GROUP BY 1,2
""")

# ---------- vasopressors (unit-aware) ----------
con.execute(f"""
CREATE TABLE inf AS
SELECT stay_id, {H('t')} AS hr,
  CASE WHEN drugname LIKE 'norepinephrine%' OR drugname LIKE '%levophed%' THEN 'ne'
       WHEN drugname LIKE 'epinephrine%' THEN 'epi'
       WHEN drugname LIKE 'dopamine%' THEN 'dopa'
       WHEN drugname LIKE 'vasopressin%' THEN 'vaso'
       WHEN drugname LIKE 'phenylephrine%' OR drugname LIKE '%neosynephrine%' THEN 'phen'
       WHEN drugname LIKE 'dobutamine%' THEN 'dobu' END AS drug,
  CASE WHEN drugname LIKE '%(mcg/kg/min)%' THEN rate
       WHEN drugname LIKE '%(mcg/min)%' THEN rate / COALESCE(NULLIF(weight,0), 80) END AS dose
FROM {R('eicu_infusion_sub')} WHERE rate > 0
""")
con.execute("""
CREATE TABLE h_vaso AS
SELECT stay_id, hr,
  MAX(CASE WHEN drug IN ('ne','epi') THEN COALESCE(dose, 0.05) END) AS ne_epi,
  MAX(CASE WHEN drug='dopa' THEN COALESCE(dose, 1) END) AS dopa,
  MAX(CASE WHEN drug IN ('vaso','phen') THEN 1 END) AS other_p,
  MAX(CASE WHEN drug='dobu' THEN 1 END) AS dobu,
  MAX(CASE WHEN drug IN ('ne','epi','dopa','vaso','phen') THEN 1 END) AS any_pressor
FROM inf WHERE drug IS NOT NULL GROUP BY 1,2
""")

# ---------- hourly SOFA (same scoring as MIMIC) ----------
con.execute("""
CREATE TABLE sofa AS
WITH a AS (
  SELECT g.stay_id, g.hr, m.map_min, gc.gcs, l.pao2, l.plt, l.bili, l.creat, l.lactate, l.fio2_lab AS fio2,
         coalesce(v.ne_epi,0) AS ne_epi, coalesce(v.dopa,0) AS dopa, coalesce(v.other_p,0) AS other_p,
         coalesce(v.dobu,0) AS dobu, coalesce(v.any_pressor,0) AS any_pressor
  FROM grid g LEFT JOIN h_map m USING (stay_id, hr) LEFT JOIN h_gcs gc USING (stay_id, hr)
       LEFT JOIN h_lab l USING (stay_id, hr) LEFT JOIN h_vaso v USING (stay_id, hr)
), b AS (
  SELECT *, pao2 / coalesce(last_value(fio2 IGNORE NULLS) OVER
        (PARTITION BY stay_id ORDER BY hr ROWS BETWEEN 4 PRECEDING AND CURRENT ROW), 0.21) AS pf FROM a
), w AS (
  SELECT stay_id, hr, lactate, any_pressor,
    MIN(pf) OVER win AS pf_min, MIN(plt) OVER win AS plt_min, MAX(bili) OVER win AS bili_max,
    MAX(creat) OVER win AS creat_max, MIN(map_min) OVER win AS map24, MIN(gcs) OVER win AS gcs_min,
    MAX(ne_epi) OVER win AS ne_epi,

In [ ]:
!pip install -q duckdb
import duckdb, pandas as pd
con = duckdb.connect()
con.execute("SET memory_limit='8GB'")
con.execute("SET threads=2")
OUT = '/content/drive/MyDrive/sepsis_project'
R = lambda f: f"read_parquet('{OUT}/{f}.parquet')"
H = lambda c: f"CAST(floor({c}/60.0) AS INTEGER)"

con.execute(f"CREATE TABLE cohort AS SELECT * FROM {R('cohort_eicu')}")
con.execute(f"CREATE TABLE grid AS SELECT stay_id, unnest(generate_series(-24, LEAST({H('outoffset')}, 168))) AS hr FROM cohort")

con.execute(f"""
CREATE TABLE vit AS
SELECT stay_id, t, heartrate AS hrt, respiration AS rrt, sao2 AS spo2t, temperature AS tempt,
       systemicsystolic AS sbpt, systemicdiastolic AS dbpt, systemicmean AS mapt FROM {R('eicu_vitalp_sub')}
UNION ALL
SELECT stay_id, t, NULL, NULL, NULL, NULL, noninvasivesystolic, noninvasivediastolic, noninvasivemean
FROM {R('eicu_vitala_sub')}
""")
con.execute(f"CREATE TABLE h_map AS SELECT stay_id, {H('t')} AS hr, MIN(mapt) AS map_min FROM vit GROUP BY 1,2")

con.execute(f"CREATE TABLE lab AS SELECT * FROM {R('eicu_lab_sub')}")
con.execute(f"""
CREATE TABLE h_lab AS
SELECT stay_id, {H('t')} AS hr,
 MIN(CASE WHEN labname='pao2' THEN v END) AS pao2,
 MAX(CASE WHEN labname='fio2' THEN CASE WHEN v>1 THEN v/100.0 ELSE NULLIF(v,0) END END) AS fio2_lab,
 MIN(CASE WHEN labname='platelets x 1000' THEN v END) AS plt,
 MAX(CASE WHEN labname='total bilirubin' THEN v END) AS bili,
 MAX(CASE WHEN labname='creatinine' THEN v END) AS creat,
 MAX(CASE WHEN labname='lactate' THEN v END) AS lactate
FROM lab GROUP BY 1,2
""")

con.execute(f"""
CREATE TABLE h_gcs AS
SELECT stay_id, {H('t')} AS hr, MIN(v) AS gcs
FROM {R('eicu_gcs_sub')} WHERE valname='GCS Total' AND v BETWEEN 3 AND 15 GROUP BY 1,2
""")

con.execute(f"""
CREATE TABLE inf AS
SELECT stay_id, {H('t')} AS hr,
  CASE WHEN drugname LIKE 'norepinephrine%' OR drugname LIKE '%levophed%' THEN 'ne'
       WHEN drugname LIKE 'epinephrine%' THEN 'epi'
       WHEN drugname LIKE 'dopamine%' THEN 'dopa'
       WHEN drugname LIKE 'vasopressin%' THEN 'vaso'
       WHEN drugname LIKE 'phenylephrine%' OR drugname LIKE '%neosynephrine%' THEN 'phen'
       WHEN drugname LIKE 'dobutamine%' THEN 'dobu' END AS drug,
  CASE WHEN drugname LIKE '%(mcg/kg/min)%' THEN rate
       WHEN drugname LIKE '%(mcg/min)%' THEN rate / COALESCE(NULLIF(weight,0), 80) END AS dose
FROM {R('eicu_infusion_sub')} WHERE rate > 0
""")
con.execute("""
CREATE TABLE h_vaso AS
SELECT stay_id, hr,
  MAX(CASE WHEN drug IN ('ne','epi') THEN COALESCE(dose, 0.05) END) AS ne_epi,
  MAX(CASE WHEN drug='dopa' THEN COALESCE(dose, 1) END) AS dopa,
  MAX(CASE WHEN drug IN ('vaso','phen') THEN 1 END) AS other_p,
  MAX(CASE WHEN drug='dobu' THEN 1 END) AS dobu,
  MAX(CASE WHEN drug IN ('ne','epi','dopa','vaso','phen') THEN 1 END) AS any_pressor
FROM inf WHERE drug IS NOT NULL GROUP BY 1,2
""")
print("Cell 1 done")

In [ ]:
con.execute("""
CREATE TABLE sofa AS
WITH a AS (
  SELECT g.stay_id, g.hr, m.map_min, gc.gcs, l.pao2, l.plt, l.bili, l.creat, l.lactate, l.fio2_lab AS fio2,
         coalesce(v.ne_epi,0) AS ne_epi, coalesce(v.dopa,0) AS dopa, coalesce(v.other_p,0) AS other_p,
         coalesce(v.dobu,0) AS dobu, coalesce(v.any_pressor,0) AS any_pressor
  FROM grid g LEFT JOIN h_map m USING (stay_id, hr) LEFT JOIN h_gcs gc USING (stay_id, hr)
       LEFT JOIN h_lab l USING (stay_id, hr) LEFT JOIN h_vaso v USING (stay_id, hr)
), b AS (
  SELECT *, pao2 / coalesce(last_value(fio2 IGNORE NULLS) OVER
        (PARTITION BY stay_id ORDER BY hr ROWS BETWEEN 4 PRECEDING AND CURRENT ROW), 0.21) AS pf FROM a
), w AS (
  SELECT stay_id, hr, lactate, any_pressor,
    MIN(pf) OVER win AS pf_min, MIN(plt) OVER win AS plt_min, MAX(bili) OVER win AS bili_max,
    MAX(creat) OVER win AS creat_max, MIN(map_min) OVER win AS map24, MIN(gcs) OVER win AS gcs_min,
    MAX(ne_epi) OVER win AS ne_epi, MAX(dopa) OVER win AS dopa, MAX(other_p) OVER win AS other_p, MAX(dobu) OVER win AS dobu
  FROM b WINDOW win AS (PARTITION BY stay_id ORDER BY hr ROWS BETWEEN 23 PRECEDING AND CURRENT ROW)
), s AS (
  SELECT *,
    CASE WHEN pf_min<100 THEN 4 WHEN pf_min<200 THEN 3 WHEN pf_min<300 THEN 2 WHEN pf_min<400 THEN 1 ELSE 0 END AS s_resp,
    CASE WHEN plt_min<20 THEN 4 WHEN plt_min<50 THEN 3 WHEN plt_min<100 THEN 2 WHEN plt_min<150 THEN 1 ELSE 0 END AS s_coag,
    CASE WHEN bili_max>=12 THEN 4 WHEN bili_max>=6 THEN 3 WHEN bili_max>=2 THEN 2 WHEN bili_max>=1.2 THEN 1 ELSE 0 END AS s_liver,
    CASE WHEN ne_epi>0.1 OR dopa>15 THEN 4 WHEN ne_epi>0 OR dopa>5 OR other_p=1 THEN 3
         WHEN dopa>0 OR dobu=1 THEN 2 WHEN map24<70 THEN 1 ELSE 0 END AS s_cv,
    CASE WHEN gcs_min<6 THEN 4 WHEN gcs_min<=9 THEN 3 WHEN gcs_min<=12 THEN 2 WHEN gcs_min<=14 THEN 1 ELSE 0 END AS s_cns,
    CASE WHEN creat_max>=5 THEN 4 WHEN creat_max>=3.5 THEN 3 WHEN creat_max>=2 THEN 2 WHEN creat_max>=1.2 THEN 1 ELSE 0 END AS s_renal
  FROM w)
SELECT *, s_resp+s_coag+s_liver+s_cv+s_cns+s_renal AS sofa FROM s
""")
con.execute(f"COPY sofa TO '{OUT}/sofa_hourly_eicu.parquet' (FORMAT PARQUET)")

con.execute(f"""
CREATE TABLE sepsis3 AS
SELECT k.*, {H('k.si_t')} AS si_hr, MIN(s.hr) AS sofa2_hr
FROM cohort k JOIN sofa s ON s.stay_id = k.stay_id
 AND s.hr BETWEEN {H('k.si_t')} - 48 AND {H('k.si_t')} + 24 AND s.sofa >= 2
GROUP BY ALL
""")

con.execute("""
CREATE TABLE shock AS
SELECT stay_id, MIN(hr) AS shock_hr FROM (
  SELECT stay_id, hr, any_pressor,
         MAX(lactate) OVER (PARTITION BY stay_id ORDER BY hr ROWS BETWEEN 6 PRECEDING AND 6 FOLLOWING) AS lac_win
  FROM sofa) WHERE any_pressor = 1 AND lac_win > 2 GROUP BY stay_id
""")

con.execute(f"""
CREATE TABLE cf AS
SELECT *, CASE WHEN shock_hr IS NOT NULL THEN 1 ELSE 0 END AS shock,
       shock_hr - index_hr AS hours_to_shock, (index_hr + 1) * 60 AS pred_min
FROM (
  SELECT s.*, k.shock_hr, GREATEST(s.si_hr, s.sofa2_hr, 0) AS index_hr
  FROM sepsis3 s LEFT JOIN shock k USING (stay_id))
WHERE index_hr < {H('outoffset')} AND (shock_hr IS NULL OR shock_hr > index_hr)
""")
print(con.execute("""
SELECT (SELECT COUNT(*) FROM cohort) AS suspected_infection,
       (SELECT COUNT(*) FROM sepsis3) AS sepsis3,
       (SELECT COUNT(*) FROM cf) AS final_cohort,
       (SELECT SUM(shock) FROM cf) AS developed_shock
""").df().T)

In [ ]:
vars_ = {'hr':'hrt','rr':'rrt','spo2':'spo2t','sbp':'sbpt','dbp':'dbpt','map':'mapt','temp':'tempt'}
aggs = []
for k, c in vars_.items():
    aggs += [f"AVG({c}) AS {k}_mean", f"MIN({c}) AS {k}_min", f"MAX({c}) AS {k}_max",
             f"arg_max({c}, t) FILTER (WHERE {c} IS NOT NULL) AS {k}_last"]
con.execute(f"""CREATE TABLE chart_feat AS
SELECT cf.stay_id, {', '.join(aggs)} FROM vit JOIN cf ON cf.stay_id = vit.stay_id
 AND vit.t < cf.pred_min AND vit.t >= cf.pred_min - 1440 GROUP BY cf.stay_id""")

labmap = {'creat':'creatinine','bili':'total bilirubin','plt':'platelets x 1000','lactate':'lactate','wbc':'wbc x 1000',
          'hgb':'hgb','k':'potassium','na':'sodium','bicarb':'bicarbonate','ph':'ph','pao2':'pao2','base_excess':'base excess'}
aggs = []
for k, nm in labmap.items():
    aggs += [f"arg_max(v, t) FILTER (WHERE labname='{nm}') AS {k}_last",
             f"MIN(v) FILTER (WHERE labname='{nm}') AS {k}_min", f"MAX(v) FILTER (WHERE labname='{nm}') AS {k}_max"]
con.execute(f"""CREATE TABLE lab_feat AS
SELECT cf.stay_id, {', '.join(aggs)} FROM lab JOIN cf ON cf.stay_id = lab.stay_id
 AND lab.t < cf.pred_min AND lab.t >= cf.pred_min - 1440 GROUP BY cf.stay_id""")

con.execute("""CREATE TABLE sofa_idx AS
SELECT cf.stay_id, s.sofa AS sofa_total, s.s_resp, s.s_coag, s.s_liver, s.s_cv, s.s_cns, s.s_renal,
       s.gcs_min, s.pf_min, s.any_pressor AS on_pressor
FROM cf JOIN sofa s ON s.stay_id = cf.stay_id AND s.hr = cf.index_hr""")

con.execute("""
CREATE TABLE features AS
SELECT cf.stay_id, cf.hadm_id, cf.hospitalid, cf.race, cf.gender, cf.hospitaldischargestatus,
       cf.age_num AS age, (cf.gender = 'Male')::INT AS male, cf.index_hr + 1 AS hours_icu_to_pred,
       cf.shock, cf.hours_to_shock,
       ch.* EXCLUDE (stay_id), lb.* EXCLUDE (stay_id), so.* EXCLUDE (stay_id)
FROM cf LEFT JOIN chart_feat ch USING (stay_id) LEFT JOIN lab_feat lb USING (stay_id) LEFT JOIN sofa_idx so USING (stay_id)
""")
con.execute(f"COPY features TO '{OUT}/features_eicu.parquet' (FORMAT PARQUET)")

df = con.execute("SELECT * FROM features").df()
print("Cohort:", len(df), "stays |", df.hospitalid.nunique(), "hospitals | shock rate:", round(100*df.shock.mean(), 1), "%")
print("Shock within 6h:", int(((df.hours_to_shock <= 6) & (df.shock == 1)).sum()),
      "| within 24h:", int(((df.hours_to_shock <= 24) & (df.shock == 1)).sum()))
print("\nShock rate by SOFA at index:")
print(df.assign(sofa_bin=pd.cut(df.sofa_total, [-1,1,3,5,7,9,11,30])).groupby('sofa_bin', observed=True).shock.agg(['mean','size']).round(3))
print("\nIn-hospital death % by shock:")
print(df.groupby('shock').hospitaldischargestatus.apply(lambda s: round(100*(s == 'Expired').mean(), 1)))
print("\nMissing % (top 8):")
print((df.isna().mean()*100).round(0).sort_values(ascending=False).head(8).to_string())

In [ ]:
vars_ = {'hr':'hrt','rr':'rrt','spo2':'spo2t','sbp':'sbpt','dbp':'dbpt','map':'mapt','temp':'tempt'}
aggs = []
for k, c in vars_.items():
    aggs += [f"AVG({c}) AS {k}_mean", f"MIN({c}) AS {k}_min", f"MAX({c}) AS {k}_max",
             f"arg_max({c}, t) FILTER (WHERE {c} IS NOT NULL) AS {k}_last"]
con.execute(f"""CREATE TABLE chart_feat AS
SELECT cf.stay_id, {', '.join(aggs)} FROM vit JOIN cf ON cf.stay_id = vit.stay_id
 AND vit.t < cf.pred_min AND vit.t >= cf.pred_min - 1440 GROUP BY cf.stay_id""")

labmap = {'creat':'creatinine','bili':'total bilirubin','plt':'platelets x 1000','lactate':'lactate','wbc':'wbc x 1000',
          'hgb':'hgb','k':'potassium','na':'sodium','bicarb':'bicarbonate','ph':'ph','pao2':'pao2','base_excess':'base excess'}
aggs = []
for k, nm in labmap.items():
    aggs += [f"arg_max(v, t) FILTER (WHERE labname='{nm}') AS {k}_last",
             f"MIN(v) FILTER (WHERE labname='{nm}') AS {k}_min", f"MAX(v) FILTER (WHERE labname='{nm}') AS {k}_max"]
con.execute(f"""CREATE TABLE lab_feat AS
SELECT cf.stay_id, {', '.join(aggs)} FROM lab JOIN cf ON cf.stay_id = lab.stay_id
 AND lab.t < cf.pred_min AND lab.t >= cf.pred_min - 1440 GROUP BY cf.stay_id""")

con.execute("""CREATE TABLE sofa_idx AS
SELECT cf.stay_id, s.sofa AS sofa_total, s.s_resp, s.s_coag, s.s_liver, s.s_cv, s.s_cns, s.s_renal,
       s.gcs_min, s.pf_min, s.any_pressor AS on_pressor
FROM cf JOIN sofa s ON s.stay_id = cf.stay_id AND s.hr = cf.index_hr""")

con.execute("""
CREATE TABLE features AS
SELECT cf.stay_id, cf.hadm_id, cf.hospitalid, cf.race, cf.gender, cf.hospitaldischargestatus,
       cf.age_num AS age, (cf.gender = 'Male')::INT AS male, cf.index_hr + 1 AS hours_icu_to_pred,
       cf.shock, cf.hours_to_shock,
       ch.* EXCLUDE (stay_id), lb.* EXCLUDE (stay_id), so.* EXCLUDE (stay_id)
FROM cf LEFT JOIN chart_feat ch USING (stay_id) LEFT JOIN lab_feat lb USING (stay_id) LEFT JOIN sofa_idx so USING (stay_id)
""")
con.execute(f"COPY features TO '{OUT}/features_eicu.parquet' (FORMAT PARQUET)")

df = con.execute("SELECT * FROM features").df()
print("Cohort:", len(df), "stays |", df.hospitalid.nunique(), "hospitals | shock rate:", round(100*df.shock.mean(), 1), "%")
print("Shock within 6h:", int(((df.hours_to_shock <= 6) & (df.shock == 1)).sum()),
      "| within 24h:", int(((df.hours_to_shock <= 24) & (df.shock == 1)).sum()))
print("\nShock rate by SOFA at index:")
print(df.assign(sofa_bin=pd.cut(df.sofa_total, [-1,1,3,5,7,9,11,30])).groupby('sofa_bin', observed=True).shock.agg(['mean','size']).round(3))
print("\nIn-hospital death % by shock:")
print(df.groupby('shock').hospitaldischargestatus.apply(lambda s: round(100*(s == 'Expired').mean(), 1)))
print("\nMissing % (top 8):")
print((df.isna().mean()*100).round(0).sort_values(ascending=False).head(8).to_string())

In [ ]:
vars_ = {'hr':'hrt','rr':'rrt','spo2':'spo2t','sbp':'sbpt','dbp':'dbpt','map':'mapt','temp':'tempt'}
aggs = []
for k, c in vars_.items():
    aggs += [f"AVG({c}) AS {k}_mean", f"MIN({c}) AS {k}_min", f"MAX({c}) AS {k}_max",
             f"arg_max({c}, t) FILTER (WHERE {c} IS NOT NULL) AS {k}_last"]
con.execute(f"""CREATE TABLE chart_feat AS
SELECT cf.stay_id, {', '.join(aggs)} FROM vit JOIN cf ON cf.stay_id = vit.stay_id
 AND vit.t < cf.pred_min AND vit.t >= cf.pred_min - 1440 GROUP BY cf.stay_id""")

labmap = {'creat':'creatinine','bili':'total bilirubin','plt':'platelets x 1000','lactate':'lactate','wbc':'wbc x 1000',
          'hgb':'hgb','k':'potassium','na':'sodium','bicarb':'bicarbonate','ph':'ph','pao2':'pao2','base_excess':'base excess'}
aggs = []
for k, nm in labmap.items():
    aggs += [f"arg_max(v, t) FILTER (WHERE labname='{nm}') AS {k}_last",
             f"MIN(v) FILTER (WHERE labname='{nm}') AS {k}_min", f"MAX(v) FILTER (WHERE labname='{nm}') AS {k}_max"]
con.execute(f"""CREATE TABLE lab_feat AS
SELECT cf.stay_id, {', '.join(aggs)} FROM lab JOIN cf ON cf.stay_id = lab.stay_id
 AND lab.t < cf.pred_min AND lab.t >= cf.pred_min - 1440 GROUP BY cf.stay_id""")

con.execute("""CREATE TABLE sofa_idx AS
SELECT cf.stay_id, s.sofa AS sofa_total, s.s_resp, s.s_coag, s.s_liver, s.s_cv, s.s_cns, s.s_renal,
       s.gcs_min, s.pf_min, s.any_pressor AS on_pressor
FROM cf JOIN sofa s ON s.stay_id = cf.stay_id AND s.hr = cf.index_hr""")

con.execute("""
CREATE TABLE features AS
SELECT cf.stay_id, cf.hadm_id, cf.hospitalid, cf.race, cf.gender, cf.hospitaldischargestatus,
       cf.age_num AS age, (cf.gender = 'Male')::INT AS male, cf.index_hr + 1 AS hours_icu_to_pred,
       cf.shock, cf.hours_to_shock,
       ch.* EXCLUDE (stay_id), lb.* EXCLUDE (stay_id), so.* EXCLUDE (stay_id)
FROM cf LEFT JOIN chart_feat ch USING (stay_id) LEFT JOIN lab_feat lb USING (stay_id) LEFT JOIN sofa_idx so USING (stay_id)
""")
con.execute(f"COPY features TO '{OUT}/features_eicu.parquet' (FORMAT PARQUET)")

df = con.execute("SELECT * FROM features").df()
print("Cohort:", len(df), "stays |", df.hospitalid.nunique(), "hospitals | shock rate:", round(100*df.shock.mean(), 1), "%")
print("Shock within 6h:", int(((df.hours_to_shock <= 6) & (df.shock == 1)).sum()),
      "| within 24h:", int(((df.hours_to_shock <= 24) & (df.shock == 1)).sum()))
print("\nShock rate by SOFA at index:")
print(df.assign(sofa_bin=pd.cut(df.sofa_total, [-1,1,3,5,7,9,11,30])).groupby('sofa_bin', observed=True).shock.agg(['mean','size']).round(3))
print("\nIn-hospital death % by shock:")
print(df.groupby('shock').hospitaldischargestatus.apply(lambda s: round(100*(s == 'Expired').mean(), 1)))
print("\nMissing % (top 8):")
print((df.isna().mean()*100).round(0).sort_values(ascending=False).head(8).to_string())

In [ ]:
for t in ['chart_feat', 'lab_feat', 'sofa_idx', 'features']:
    con.execute(f"DROP TABLE IF EXISTS {t}")

vars_ = {'hr':'hrt','rr':'rrt','spo2':'spo2t','sbp':'sbpt','dbp':'dbpt','map':'mapt','temp':'tempt'}
aggs = []
for k, c in vars_.items():
    aggs += [f"AVG({c}) AS {k}_mean", f"MIN({c}) AS {k}_min", f"MAX({c}) AS {k}_max",
             f"arg_max({c}, t) FILTER (WHERE {c} IS NOT NULL) AS {k}_last"]
con.execute(f"""CREATE TABLE chart_feat AS
SELECT cf.stay_id, {', '.join(aggs)} FROM vit JOIN cf ON cf.stay_id = vit.stay_id
 AND vit.t < cf.pred_min AND vit.t >= cf.pred_min - 1440 GROUP BY cf.stay_id""")

labmap = {'creat':'creatinine','bili':'total bilirubin','plt':'platelets x 1000','lactate':'lactate','wbc':'wbc x 1000',
          'hgb':'hgb','k':'potassium','na':'sodium','bicarb':'bicarbonate','ph':'ph','pao2':'pao2','base_excess':'base excess'}
aggs = []
for k, nm in labmap.items():
    aggs += [f"arg_max(v, t) FILTER (WHERE labname='{nm}') AS {k}_last",
             f"MIN(v) FILTER (WHERE labname='{nm}') AS {k}_min", f"MAX(v) FILTER (WHERE labname='{nm}') AS {k}_max"]
con.execute(f"""CREATE TABLE lab_feat AS
SELECT cf.stay_id, {', '.join(aggs)} FROM lab JOIN cf ON cf.stay_id = lab.stay_id
 AND lab.t < cf.pred_min AND lab.t >= cf.pred_min - 1440 GROUP BY cf.stay_id""")

con.execute("""CREATE TABLE sofa_idx AS
SELECT cf.stay_id, s.sofa AS sofa_total, s.s_resp, s.s_coag, s.s_liver, s.s_cv, s.s_cns, s.s_renal,
       s.gcs_min, s.pf_min, s.any_pressor AS on_pressor
FROM cf JOIN sofa s ON s.stay_id = cf.stay_id AND s.hr = cf.index_hr""")

con.execute("""
CREATE TABLE features AS
SELECT cf.stay_id, cf.hadm_id, cf.hospitalid, cf.race, cf.gender, cf.hospitaldischargestatus,
       cf.age_num AS age, (cf.gender = 'Male')::INT AS male, cf.index_hr + 1 AS hours_icu_to_pred,
       cf.shock, cf.hours_to_shock,
       ch.* EXCLUDE

In [ ]:
for t in ['chart_feat', 'lab_feat', 'sofa_idx', 'features']:
    con.execute(f"DROP TABLE IF EXISTS {t}")

vars_ = {'hr':'hrt','rr':'rrt','spo2':'spo2t','sbp':'sbpt','dbp':'dbpt','map':'mapt','temp':'tempt'}
aggs = []
for k, c in vars_.items():
    aggs += [f"AVG({c}) AS {k}_mean", f"MIN({c}) AS {k}_min", f"MAX({c}) AS {k}_max",
             f"arg_max({c}, t) FILTER (WHERE {c} IS NOT NULL) AS {k}_last"]
q = "CREATE TABLE chart_feat AS SELECT cf.stay_id, " + ", ".join(aggs)
q += " FROM vit JOIN cf ON cf.stay_id = vit.stay_id AND vit.t < cf.pred_min AND vit.t >= cf.pred_min - 1440 GROUP BY cf.stay_id"
con.execute(q)

labmap = {'creat':'creatinine','bili':'total bilirubin','plt':'platelets x 1000','lactate':'lactate','wbc':'wbc x 1000','hgb':'hgb','k':'potassium','na':'sodium','bicarb':'bicarbonate','ph':'ph','pao2':'pao2','base_excess':'base excess'}
aggs = []
for k, nm in labmap.items():
    aggs += [f"arg_max(v, t) FILTER (WHERE labname='{nm}') AS {k}_last",
             f"MIN(v) FILTER (WHERE labname='{nm}') AS {k}_min",
             f"MAX(v) FILTER (WHERE labname='{nm}') AS {k}_max"]
q = "CREATE TABLE lab_feat AS SELECT cf.stay_id, " + ", ".join(aggs)
q += " FROM lab JOIN cf ON cf.stay_id = lab.stay_id AND lab.t < cf.pred_min AND lab.t >= cf.pred_min - 1440 GROUP BY cf.stay_id"
con.execute(q)
print("Cell 3a done")

In [ ]:
q = "CREATE TABLE sofa_idx AS SELECT cf.stay_id, s.sofa AS sofa_total, s.s_resp, s.s_coag, s.s_liver, s.s_cv, s.s_cns, s.s_renal, s.gcs_min, s.pf_min, s.any_pressor AS on_pressor"
q += " FROM cf JOIN sofa s ON s.stay_id = cf.stay_id AND s.hr = cf.index_hr"
con.execute(q)

q = "CREATE TABLE features AS SELECT cf.stay_id, cf.hadm_id, cf.hospitalid, cf.race, cf.gender, cf.hospitaldischargestatus, "
q += "cf.age_num AS age, (cf.gender = 'Male')::INT AS male, cf.index_hr + 1 AS hours_icu_to_pred, cf.shock, cf.hours_to_shock, "
q += "ch.* EXCLUDE (stay_id), lb.* EXCLUDE (stay_id), so.* EXCLUDE (stay_id) "
q += "FROM cf LEFT JOIN chart_feat ch USING (stay_id) LEFT JOIN lab_feat lb USING (stay_id) LEFT JOIN sofa_idx so USING (stay_id)"
con.execute(q)

con.execute(f"COPY features TO '{OUT}/features_eicu.parquet' (FORMAT PARQUET)")
print("Cell 3b done")

In [ ]:
df = con.execute("SELECT * FROM features").df()
print("Cohort:", len(df), "stays |", df.hospitalid.nunique(), "hospitals | shock rate:", round(100*df.shock.mean(), 1), "%")
print("Shock within 6h:", int(((df.hours_to_shock <= 6) & (df.shock == 1)).sum()),
      "| within 24h:", int(((df.hours_to_shock <= 24) & (df.shock == 1)).sum()))
print("\nShock rate by SOFA at index:")
print(df.assign(sofa_bin=pd.cut(df.sofa_total, [-1,1,3,5,7,9,11,30])).groupby('sofa_bin', observed=True).shock.agg(['mean','size']).round(3))
print("\nIn-hospital death % by shock:")
print(df.groupby('shock').hospitaldischargestatus.apply(lambda s: round(100*(s == 'Expired').mean(), 1)))
print("\nMissing % (top 8):")
print((df.isna().mean()*100).round(0).sort_values(ascending=False).head(8).to_string())

In [ ]:
import pandas as pd, numpy as np, joblib
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

OUT = '/content/drive/MyDrive/sepsis_project'
saved = joblib.load(f'{OUT}/rf_mimic_model.joblib')
rf, feats = saved['model'], saved['feats']

ev = pd.read_parquet(f'{OUT}/features_eicu.parquet')
num = ev.select_dtypes(include=[np.number]).columns
ev[num] = ev[num].replace([np.inf, -np.inf], np.nan)
limits = {'hr':(0,300),'rr':(0,100),'spo2':(0,100),'sbp':(0,350),'dbp':(0,250),'map':(0,300),'temp':(25,45)}
for c in ev.columns:
    for key,(lo,hi) in limits.items():
        if c.startswith(key + '_'):
            ev.loc[(ev[c] < lo) | (ev[c] > hi), c] = np.nan
ev.loc[ev['pf_min'] > 800, 'pf_min'] = np.nan
for c in feats:
    if c not in ev.columns:
        ev[c] = np.nan
print("Features missing in eICU and filled empty:", [c for c in feats if ev[c].isna().all()])

y = ev.shock.values
p_rf = rf.predict_proba(ev[feats])[:, 1]
p_sofa = ev['sofa_total'].fillna(0).values.astype(float)

# cut-off from MIMIC test set at 80% specificity
m = pd.read_parquet(f'{OUT}/preds_rf_mimic_test.parquet')
fpr, tpr, thr = roc_curve(m.shock, m.p_rf)
i = np.searchsorted(fpr, 0.20, side='right') - 1
cut = thr[i]

def sens_at_spec(y, p, spec):
    fpr, tpr, _ = roc_curve(y, p)
    return tpr[np.searchsorted(fpr, 1 - spec, side='right') - 1]

def report(name, y, p, n_boot=200, seed=0):
    rng = np.random.default_rng(seed)
    s = {'AUROC': [], 'AUPRC': [], 'Sens@80%spec': [], 'Sens@90%spec': []}
    for _ in range(n_boot):
        k = rng.choice(len(y), len(y))
        if y[k].sum() == 0: continue
        s['AUROC'].append(roc_auc_score(y[k], p[k]))
        s['AUPRC'].append(average_precision_score(y[k], p[k]))
        s['Sens@80%spec'].append(sens_at_spec(y[k], p[k], 0.80))
        s['Sens@90%spec'].append(sens_at_spec(y[k], p[k], 0.90))
    out = {'model': name}
    for k, v in s.items():
        out[k] = f"{np.mean(v):.3f} ({np.percentile(v,2.5):.3f}-{np.percentile(v,97.5):.3f})"
    return out

print(pd.DataFrame([report('RF trained on MIMIC, tested on eICU', y, p_rf),
                    report('SOFA only', y, p_sofa)]).to_string(index=False))
print(f"\nPrevalence (random-model AUPRC) = {y.mean():.3f}")

pred = p_rf >= cut
print(f"\nMIMIC cut-off {cut:.3f} applied to eICU: sensitivity {pred[y==1].mean():.3f}, specificity {1-pred[y==0].mean():.3f}")

ev[['stay_id','shock','race','gender','age','hospitalid']].assign(p_rf=p_rf).to_parquet(f'{OUT}/preds_rf_eicu.parquet')

In [ ]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

OUT = '/content/drive/MyDrive/sepsis_project'

def clean(d):
    num = d.select_dtypes(include=[np.number]).columns
    d[num] = d[num].replace([np.inf, -np.inf], np.nan)
    lim = {'hr':(0,300),'rr':(0,100),'spo2':(0,100),'sbp':(0,350),'dbp':(0,250),'map':(0,300),'temp':(25,45)}
    for c in d.columns:
        for k, (lo, hi) in lim.items():
            if c.startswith(k + '_'):
                d.loc[(d[c] < lo) | (d[c] > hi), c] = np.nan
    d.loc[d['pf_min'] > 800, 'pf_min'] = np.nan
    return d

mi = clean(pd.read_parquet(f'{OUT}/features_mimic.parquet'))
ev = clean(pd.read_parquet(f'{OUT}/features_eicu.parquet'))
groups = ['2008 - 2010', '2011 - 2013', '2014 - 2016']
train = mi[mi.anchor_year_group.isin(groups)]

drop = ['stay_id','subject_id','hadm_id','anchor_year_group','race','gender','shock','hours_to_shock']
feats = [c for c in mi.columns if c not in drop and not c.startswith('temp_')]
print(len(feats), "features (temperature removed)")

m = make_pipeline(SimpleImputer(strategy='median', add_indicator=True),
    RandomForestClassifier(n_estimators=500, min_samples_leaf=5,
    class_weight='balanced_subsample', n_jobs=-1, random_state=42))
m.fit(train[feats], train.shock)
p = m.predict_proba(ev[feats])[:, 1]
print("eICU without temperature: AUROC", round(roc_auc_score(ev.shock, p), 3),
      "| AUPRC", round(average_precision_score(ev.shock, p), 3))

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

OUT = '/content/drive/MyDrive/sepsis_project'
mi = pd.read_parquet(f'{OUT}/features_mimic.parquet')
co = pd.read_parquet(f'{OUT}/cohort_final_mimic.parquet')[['stay_id','hospital_expire_flag']]
mi = mi.merge(co, on='stay_id')
mi['died'] = mi.hospital_expire_flag
ev = pd.read_parquet(f'{OUT}/features_eicu.parquet')
ev['died'] = (ev.hospitaldischargestatus == 'Expired').astype(int)

def med(s):
    return f"{s.median():.1f} ({s.quantile(0.25):.1f}-{s.quantile(0.75):.1f})"

def col(d):
    return {
        'Stays, n': f"{len(d):,}",
        'Age, median (IQR)': med(d.age),
        'Male, %': f"{100*d.male.mean():.1f}",
        'SOFA at index, median (IQR)': med(d.sofa_total),
        'Last lactate, median (IQR)': med(d.lactate_last.dropna()),
        'Hours from ICU admission to index, median (IQR)': med(d.hours_icu_to_pred),
        'Septic shock after index, n (%)': f"{int(d.shock.sum()):,} ({100*d.shock.mean():.1f})",
        'In-hospital death, %': f"{100*d.died.mean():.1f}",
        'In-hospital death if shock, %': f"{100*d[d.shock==1].died.mean():.1f}",
        'In-hospital death if no shock, %': f"{100*d[d.shock==0].died.mean():.1f}",
    }

t1 = pd.DataFrame({'MIMIC-IV': col(mi), 'eICU-CRD': col(ev)})
t1.to_csv(f'{OUT}/table1_cohorts.csv')
print(t1.to_string())

pm = pd.read_parquet(f'{OUT}/preds_rf_mimic_test.parquet')
pe = pd.read_parquet(f'{OUT}/preds_rf_eicu.parquet')
sofa_e = ev.set_index('stay_id').loc[pe.stay_id, 'sofa_total'].fillna(0).values

curves = [
    ('RF, MIMIC-IV temporal test', pm.shock.values, pm.p_rf.values, '-'),
    ('RF, eICU external test', pe.shock.values, pe.p_rf.values, '-'),
    ('SOFA only, eICU', pe.shock.values, sofa_e, '--'),
]
plt.figure(figsize=(6, 6))
for name, y, p, ls in curves:
    fpr, tpr, _ = roc_curve(y, p)
    plt.plot(fpr, tpr, ls, label=f"{name} (AUROC {roc_auc_score(y, p):.3f})")
plt.plot([0, 1], [0, 1], ':', color='grey')
plt.xlabel('1 - Specificity')
plt.ylabel('Sensitivity')
plt.title('Early prediction of septic shock')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUT}/roc_curves.png', dpi=200)
plt.show()